In [ ]:
# Standard library
import os
import re
import sys
import random
import warnings
from pathlib import Path
from collections import defaultdict
from itertools import product

# Numerical computing
import numpy as np
import numpy.linalg as la

# Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Polygon, PathPatch
from matplotlib.path import Path as MPath

# Data handling
from tqdm import tqdm


# Project imports
def _find_repo_root(markers=("config.py", "CITATION.cff")):
    """
    Locate the repository root by walking up from the current working
    directory until a directory containing all `markers` is found.

    More robust than `Path.cwd().parent[.parent]`: independent of which
    directory the Jupyter kernel happened to start in, and fails loudly
    (rather than silently importing the wrong `config.py`) if the notebook
    is run outside the repository.
    """
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if all((candidate / m).exists() for m in markers):
            return candidate
    raise RuntimeError(
        f"Could not locate repository root from {start}. "
        f"Expected to find {markers} in some parent directory. "
        f"Open this notebook from inside the optical-eigentask-learning "
        f"repository."
    )


REPO_ROOT = _find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "code"))

from config import (
    DATA_DIR,
    RESULT_DIR,
    MNIST_LENS_DIR,
    MPEG7_LENS_DIR,
    SPDNN_DIR,
    FIGURE_DIR,
)
from training import eigentask_solver, pca_solver, fft

In [ ]:
color_dic = {
    "pca": [254 / 255, 207 / 255, 146 / 255],
    "lpfft": [100 / 255, 26 / 255, 128 / 255],
    "eigen": [183 / 255, 55 / 255, 121 / 255],
    "cg": [247 / 255, 112 / 255, 92 / 255],
    "sy": [20 / 255, 14 / 255, 54 / 255],
    "lp": [20 / 255, 14 / 255, 54 / 255],
}

legend_dic = {
    "pca": "PCA",
    "lpfft": "Low-pass filtering",
    "eigen": "Eigentasks",
    "cg": "Coarse graining",
    "sy": r"Ma $\mathit{et}$ $\mathit{al.}$",
    "lp": "Raw data",
}
line_alpha = 70 / 255
marker_line_alpha = 255 / 255
face_alpha = 130 / 255

ls_dic = {"linear": "--", "logistic": "-", "running_max": "-"}

scale = 1

import matplotlib as mpl

scale = 1
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.titlesize": 8 * scale,
        "axes.labelsize": 8 * scale,
        "xtick.labelsize": 7 * scale,
        "ytick.labelsize": 7 * scale,
        "legend.fontsize": 7 * scale,
        "axes.linewidth": 0.7 * scale,
        "grid.linewidth": 0.5 * scale,
        "xtick.major.width": 0.6 * scale,
        "ytick.major.width": 0.6 * scale,
        "xtick.minor.width": 0.5 * scale,
        "ytick.minor.width": 0.5 * scale,
        "lines.linewidth": 0.5 * scale,
        "lines.markeredgewidth": 0.5 * scale,
        "lines.markersize": 2.5 * scale,
        "lines.marker": "o",
        "savefig.bbox": None,
        "savefig.pad_inches": 0.0 * scale,
    }
)


def adjust_fixed_margins(fig, left_in=0.55, right_in=0.08, bottom_in=0.45, top_in=0.18):
    """
    fix margins in inch: left/right/bottom/top
    """
    W, H = fig.get_size_inches()
    fig.subplots_adjust(
        left=left_in / W,
        right=1 - right_in / W,
        bottom=bottom_in / H,
        top=1 - top_in / H,
    )

In [ ]:
data_dir = DATA_DIR
training_result_dir = RESULT_DIR
fig_folder_dir = FIGURE_DIR

## Figure 1

In [ ]:
fig_dir = fig_folder_dir / "fig1"
fig_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Heights for each class (horizontal = bar length)
heights = np.array([0.15, 0.22, 0.18, 0.12, 0.20, 0.16, 0.95, 0.14, 0.19, 0.13])
n = len(heights)
highlight = 6  # Tallest bar, shown in bold

# Colors from magma — outline, fill, and labels all share one hue
cmap = plt.get_cmap("magma")
base = np.array(cmap(0.55)[:3])
line_color = tuple(base)
fill_color = tuple(0.35 * base + 0.65)
text_color = tuple(base)

fig, ax = plt.subplots(figsize=(2.4, 2.4), dpi=150)
fig.patch.set_facecolor("white")

edges = np.arange(n + 1) - 0.5

# Build the stepped vertex sequence (bottom-left -> up the right edge -> top-left)
step_verts = [(0.0, edges[0])]
for i in range(n):
    step_verts.append((heights[i], edges[i]))
    step_verts.append((heights[i], edges[i + 1]))
step_verts.append((0.0, edges[-1]))

# Closed polygon for the fill (left edge hidden via edgecolor='none')
fill_patch = Polygon(
    step_verts, closed=True, facecolor=fill_color, edgecolor="none", zorder=1
)
ax.add_patch(fill_patch)

# Open path for the outline — not closed, so the left edge is not drawn
outline_codes = [MPath.MOVETO] + [MPath.LINETO] * (len(step_verts) - 1)
outline_path = MPath(step_verts, outline_codes)
outline_patch = PathPatch(
    outline_path,
    facecolor="none",
    edgecolor=line_color,
    linewidth=1.4,
    joinstyle="miter",
    capstyle="butt",
    zorder=2,
)
ax.add_patch(outline_patch)

# Digit labels 0-9 in the same magma color
for i in range(n):
    weight = "bold" if i == highlight else "normal"
    size = 15 if i == highlight else 13
    ax.text(
        -0.06,
        i,
        str(i),
        ha="right",
        va="center",
        fontsize=size,
        fontweight=weight,
        color=text_color,
    )

# Strip axis decorations
ax.set_xlim(-0.35, max(heights) * 1.08)
ax.set_ylim(edges[0] - 0.1, edges[-1] + 0.1)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig(
    fig_dir / "output.svg",
    format="svg",
    bbox_inches="tight",
    pad_inches=0.05,
    transparent=True,
)
plt.show()

# MNIST + Lens Front-End Figures

## Load Data

In [ ]:
logistic_result_dir = RESULT_DIR / "mnist_logistic"
nn_result_dir = RESULT_DIR / "mnist_nn"

dataset_dir = {
    "high": MNIST_LENS_DIR["high"] / "frames",
    "low": MNIST_LENS_DIR["low"] / "frames",
}

preferred_linear_source = "logistic"  # "logistic" or "nn"

seed_list = [0, 1, 2, 3, 4]
power_list = ["high", "low"]
thresholded_list = [True, False]
stdz_list = [True, False]
train_mode_list = ["lpfft", "eigen", "cg", "pca"]
num_frames_list = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 40, 60, 80, 100]

verbose = True

# ====================
# Small utilities
# ====================


def vprint(*args, **kwargs):
    if verbose:
        print(*args, **kwargs)


def warn(msg):
    warnings.warn(str(msg), RuntimeWarning, stacklevel=2)


def parse_bool(x):
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, (int, np.integer)):
        return bool(int(x))
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true", "t", "1", "yes", "y"}:
            return True
        if s in {"false", "f", "0", "no", "n"}:
            return False
    raise ValueError(f"Cannot parse bool from {x!r}")


def unpack_scalar(x):
    while isinstance(x, np.ndarray) and x.ndim == 0:
        x = x.item()
    return x


def flatten_numeric_values(x):
    x = unpack_scalar(x)
    if x is None:
        return
    if isinstance(x, np.ndarray):
        if x.dtype == object:
            for item in x.tolist():
                yield from flatten_numeric_values(item)
        else:
            for item in np.asarray(x).reshape(-1):
                yield float(item)
    elif isinstance(x, (list, tuple)):
        for item in x:
            yield from flatten_numeric_values(item)
    else:
        try:
            yield float(x)
        except Exception:
            return


def to_1d_float_array(x, name="array"):
    vals = list(flatten_numeric_values(x))
    if len(vals) == 0:
        return np.array([], dtype=float)
    try:
        return np.asarray(vals, dtype=float).reshape(-1)
    except Exception as e:
        warn(f"Failed to convert {name} to 1D float array: {e}")
        return np.array([], dtype=float)


def to_float_scalar(x, default=np.nan):
    x = unpack_scalar(x)
    if x is None:
        return default
    if isinstance(x, np.ndarray):
        arr = to_1d_float_array(x, name="scalar")
        return float(arr[0]) if arr.size > 0 else default
    if isinstance(x, (list, tuple)):
        arr = to_1d_float_array(x, name="scalar")
        return float(arr[0]) if arr.size > 0 else default
    try:
        return float(x)
    except Exception:
        return default


def normalize_K_list(k_list_raw):
    arr = to_1d_float_array(k_list_raw, name="K_list")
    if arr.size == 0:
        return np.array([], dtype=int)
    if np.all(np.isfinite(arr)) and np.allclose(arr, np.round(arr), atol=1e-12, rtol=0):
        return np.round(arr).astype(int)
    return arr.astype(float)


def seq_len(x):
    x = unpack_scalar(x)
    if x is None:
        return 0
    if isinstance(x, np.ndarray):
        if x.ndim == 0:
            return 1
        return int(x.shape[0])
    if isinstance(x, (list, tuple)):
        return len(x)
    return 1


def get_k_entry(container, idx, name="container"):
    container = unpack_scalar(container)
    if container is None:
        return None
    try:
        if isinstance(container, np.ndarray):
            if container.ndim == 0:
                return container.item() if idx == 0 else None
            if idx < container.shape[0]:
                return container[idx]
            warn(f"{name}: idx {idx} out of range for shape {container.shape}")
            return None
        if isinstance(container, (list, tuple)):
            if idx < len(container):
                return container[idx]
            warn(f"{name}: idx {idx} out of range for length {len(container)}")
            return None
        return container if idx == 0 else None
    except Exception as e:
        warn(f"Failed to extract entry {idx} from {name}: {e}")
        return None


def parse_running_triplet(entry, name="running_triplet"):
    empty = (
        np.array([], dtype=float),
        np.array([], dtype=float),
        np.array([], dtype=float),
    )
    entry = unpack_scalar(entry)
    if entry is None:
        return empty

    try:
        if isinstance(entry, np.ndarray) and entry.dtype != object:
            arr = np.asarray(entry, dtype=float)
            if arr.ndim == 1:
                if arr.size == 0:
                    return empty
                if arr.size == 3:
                    # Ambiguous, but keep length-1 histories if needed.
                    return (
                        np.asarray([arr[0]], dtype=float),
                        np.asarray([arr[1]], dtype=float),
                        np.asarray([arr[2]], dtype=float),
                    )
                warn(
                    f"{name}: unexpected numeric 1D shape {arr.shape}; cannot infer triplet safely"
                )
                return empty
            if arr.ndim >= 2:
                if arr.shape[0] == 3:
                    return (
                        np.asarray(arr[0]).reshape(-1).astype(float),
                        np.asarray(arr[1]).reshape(-1).astype(float),
                        np.asarray(arr[2]).reshape(-1).astype(float),
                    )
                if arr.shape[1] == 3:
                    warn(
                        f"{name}: interpreted numeric array with shape {arr.shape} as (epoch, 3)"
                    )
                    return (
                        np.asarray(arr[:, 0]).reshape(-1).astype(float),
                        np.asarray(arr[:, 1]).reshape(-1).astype(float),
                        np.asarray(arr[:, 2]).reshape(-1).astype(float),
                    )
                warn(
                    f"{name}: numeric array shape {arr.shape} does not expose a 3-way triplet axis"
                )
                return empty

        if isinstance(entry, np.ndarray) and entry.dtype == object:
            parts = entry.tolist()
        elif isinstance(entry, (list, tuple)):
            parts = list(entry)
        else:
            warn(f"{name}: unsupported entry type {type(entry)}")
            return empty

        if len(parts) < 3:
            warn(f"{name}: expected at least 3 parts, got {len(parts)}")
            return empty

        train_hist = to_1d_float_array(parts[0], name=f"{name}[0]")
        val_hist = to_1d_float_array(parts[1], name=f"{name}[1]")
        test_hist = to_1d_float_array(parts[2], name=f"{name}[2]")
        return train_hist, val_hist, test_hist

    except Exception as e:
        warn(f"Failed to parse {name}: {e}")
        return empty


def parse_final_triplet(entry, name="final_triplet"):
    entry = unpack_scalar(entry)
    if entry is None:
        return (np.nan, np.nan, np.nan)

    try:
        if isinstance(entry, np.ndarray) and entry.dtype != object:
            arr = np.asarray(entry, dtype=float).reshape(-1)
            if arr.size >= 3:
                return (float(arr[0]), float(arr[1]), float(arr[2]))
            if arr.size == 2:
                return (float(arr[0]), np.nan, float(arr[1]))
            if arr.size == 1:
                return (np.nan, float(arr[0]), np.nan)
            return (np.nan, np.nan, np.nan)

        if isinstance(entry, np.ndarray) and entry.dtype == object:
            parts = entry.tolist()
        elif isinstance(entry, (list, tuple)):
            parts = list(entry)
        else:
            parts = [entry]

        vals = []
        for part in parts:
            arr = to_1d_float_array(part, name=name)
            if arr.size > 0:
                vals.append(float(arr[0]))
            else:
                vals.append(np.nan)

        if len(vals) >= 3:
            return (vals[0], vals[1], vals[2])
        if len(vals) == 2:
            return (vals[0], np.nan, vals[1])
        if len(vals) == 1:
            return (np.nan, vals[0], np.nan)
        return (np.nan, np.nan, np.nan)

    except Exception as e:
        warn(f"Failed to parse {name}: {e}")
        return (np.nan, np.nan, np.nan)


def safe_hist_value(hist, idx, name="hist"):
    hist = to_1d_float_array(hist, name=name)
    if hist.size == 0:
        return np.nan
    if idx < 0:
        return np.nan
    if idx >= hist.size:
        warn(f"{name}: idx {idx} out of range for length {hist.size}")
        return np.nan
    return float(hist[idx])


def select_best_epoch_from_val_acc(
    val_hist,
    train_hist=None,
    test_hist=None,
    val_loss_hist=None,
    test_loss_hist=None,
):
    val_hist = to_1d_float_array(val_hist, name="val_hist")
    if val_hist.size == 0:
        return -1

    valid_idx = np.where(np.isfinite(val_hist))[0]
    if valid_idx.size == 0:
        return -1

    max_val = np.nanmax(val_hist[valid_idx])
    tie_idx = valid_idx[np.isclose(val_hist[valid_idx], max_val, rtol=0, atol=1e-12)]
    if tie_idx.size == 0:
        return -1
    return int(tie_idx[0])  # earliest epoch


def select_best_K_from_val_acc(K_array, epoch_array, val_acc_array):
    K_array = to_1d_float_array(K_array, name="K_array")
    epoch_array = to_1d_float_array(epoch_array, name="epoch_array")
    val_acc_array = to_1d_float_array(val_acc_array, name="val_acc_array")

    n = min(len(K_array), len(epoch_array), len(val_acc_array))
    if n == 0:
        return -1

    K_array = K_array[:n]
    epoch_array = epoch_array[:n]
    val_acc_array = val_acc_array[:n]

    valid = np.isfinite(K_array) & np.isfinite(val_acc_array)
    valid_idx = np.where(valid)[0]
    if valid_idx.size == 0:
        return -1

    max_val = np.nanmax(val_acc_array[valid_idx])
    tie_idx = valid_idx[
        np.isclose(val_acc_array[valid_idx], max_val, rtol=0, atol=1e-12)
    ]

    if tie_idx.size > 1:
        min_K = np.nanmin(K_array[tie_idx])
        tie_idx = tie_idx[np.isclose(K_array[tie_idx], min_K, rtol=0, atol=1e-12)]

    if tie_idx.size > 1:
        epoch_cmp = epoch_array[tie_idx].copy()
        epoch_cmp[~np.isfinite(epoch_cmp)] = np.inf
        min_epoch = np.min(epoch_cmp)
        tie_idx = tie_idx[epoch_cmp == min_epoch]

    return int(tie_idx[0]) if tie_idx.size > 0 else -1


def nanmean_safe(a, axis=0):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(a, axis=axis)


def nanstd_safe(a, axis=0):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanstd(a, axis=axis)


# ====================
# File parsing
# ====================

FILENAME_RE = re.compile(
    r"^power_(?P<power>.+?)_thresholded_(?P<thresholded>True|False)_shots_(?P<num_frames>\d+)_Vmax_(?P<Vmax>True|False)_train_(?P<train_mode>.+?)_seed_(?P<seed>-?\d+)_stdz_(?P<stdz>True|False)\.npz$"
)


def safe_np_load(path):
    return np.load(path, allow_pickle=True)


def parse_metadata_from_filename(path):
    m = FILENAME_RE.match(Path(path).name)
    if m is None:
        return None
    d = m.groupdict()
    return {
        "power": d["power"],
        "thresholded": parse_bool(d["thresholded"]),
        "num_frames": int(d["num_frames"]),
        "Vmax": parse_bool(d["Vmax"]),
        "train_mode": d["train_mode"],
        "seed": int(d["seed"]),
        "stdz": parse_bool(d["stdz"]),
    }


def npz_get_scalar(data, key, default=None):
    if key not in data.files:
        return default
    return unpack_scalar(data[key])


def read_metadata_from_npz(data, fallback=None):
    fallback = {} if fallback is None else dict(fallback)
    meta = dict(fallback)

    try:
        if "power" in data.files:
            meta["power"] = str(npz_get_scalar(data, "power"))
        if "thresholded" in data.files:
            meta["thresholded"] = parse_bool(npz_get_scalar(data, "thresholded"))
        if "num_frames" in data.files:
            meta["num_frames"] = int(
                to_float_scalar(npz_get_scalar(data, "num_frames"))
            )
        if "train_mode" in data.files:
            meta["train_mode"] = str(npz_get_scalar(data, "train_mode"))
        if "seed" in data.files:
            meta["seed"] = int(to_float_scalar(npz_get_scalar(data, "seed")))
        if "Vmax" in data.files:
            meta["Vmax"] = parse_bool(npz_get_scalar(data, "Vmax"))
        # stdz is not guaranteed to be saved; usually parsed from filename.
    except Exception as e:
        warn(f"Failed to read metadata from npz: {e}")

    return meta


seed_filter = None if seed_list is None else set(seed_list)
power_filter = None if power_list is None else set(power_list)
thresholded_filter = None if thresholded_list is None else set(thresholded_list)
stdz_filter = None if stdz_list is None else set(stdz_list)
train_mode_filter = None if train_mode_list is None else set(train_mode_list)
num_frames_filter = None if num_frames_list is None else set(num_frames_list)


def metadata_matches_filters(meta):
    try:
        if seed_filter is not None and meta.get("seed") not in seed_filter:
            return False
        if power_filter is not None and meta.get("power") not in power_filter:
            return False
        if (
            thresholded_filter is not None
            and meta.get("thresholded") not in thresholded_filter
        ):
            return False
        if stdz_filter is not None and meta.get("stdz") not in stdz_filter:
            return False
        if (
            train_mode_filter is not None
            and meta.get("train_mode") not in train_mode_filter
        ):
            return False
        if (
            num_frames_filter is not None
            and meta.get("num_frames") not in num_frames_filter
        ):
            return False
    except Exception as e:
        warn(f"Failed metadata filter check for meta={meta}: {e}")
        return False
    return True


def preferred_source_rank(source_name):
    if preferred_linear_source not in {"logistic", "nn"}:
        raise ValueError("preferred_linear_source must be 'logistic' or 'nn'")
    if source_name == preferred_linear_source:
        return 0
    return 1


# ====================
# Scan raw run files
# ====================

raw_runs = {
    "logistic": {},
    "nn": {},
    "linear": {},
}

scan_counts = {
    "logistic_files_seen": 0,
    "nn_files_seen": 0,
    "logistic_loaded": 0,
    "nn_loaded": 0,
    "linear_loaded": 0,
}

source_dirs = [
    ("logistic", Path(logistic_result_dir)),
    ("nn", Path(nn_result_dir)),
]

for source_name, result_dir in source_dirs:
    if not result_dir.exists():
        warn(f"Result directory does not exist: {result_dir}")
        continue

    npz_paths = sorted(result_dir.glob("*.npz"))
    vprint(f"[scan] {source_name}: found {len(npz_paths)} npz files in {result_dir}")

    for path in npz_paths:
        scan_counts[f"{source_name}_files_seen"] += 1

        meta = parse_metadata_from_filename(path)
        if meta is None:
            warn(
                f"Filename did not match expected pattern, will try reading metadata from file: {path.name}"
            )

        try:
            with safe_np_load(path) as data:
                meta = read_metadata_from_npz(data, fallback=meta)

                required_keys = [
                    "power",
                    "thresholded",
                    "num_frames",
                    "train_mode",
                    "seed",
                    "stdz",
                ]
                if any(k not in meta for k in required_keys):
                    missing = [k for k in required_keys if k not in meta]
                    warn(f"Skipping file with incomplete metadata {missing}: {path}")
                    continue

                if not metadata_matches_filters(meta):
                    continue

                fileset = set(data.files)
                inferred_model_kind = None
                if "acc_logistic" in fileset:
                    inferred_model_kind = "logistic"
                elif "acc_nn" in fileset:
                    inferred_model_kind = "nn"

                if inferred_model_kind is None:
                    warn(f"Skipping file without acc_logistic / acc_nn: {path}")
                    continue

                if inferred_model_kind != source_name:
                    warn(
                        f"Model kind inferred from file ({inferred_model_kind}) disagrees with source dir label ({source_name}) for {path.name}"
                    )

                model_kind = inferred_model_kind

                K_list = normalize_K_list(data["K_list"] if "K_list" in fileset else [])
                if K_list.size == 0:
                    warn(
                        f"Empty or missing K_list in {path}; will try to infer later from container lengths"
                    )

                base_record = {
                    "path": str(path),
                    "source": source_name,
                    "model_kind": model_kind,
                    "meta": dict(meta),
                    "K_list": K_list,
                }

                if model_kind == "logistic":
                    model_record = {
                        **base_record,
                        "final_container": (
                            data["acc_logistic"] if "acc_logistic" in fileset else None
                        ),
                        "running_container": (
                            data["acc_logistic_running"]
                            if "acc_logistic_running" in fileset
                            else None
                        ),
                        "loss_container": (
                            data["acc_logistic_loss"]
                            if "acc_logistic_loss" in fileset
                            else None
                        ),
                    }
                else:
                    model_record = {
                        **base_record,
                        "final_container": (
                            data["acc_nn"] if "acc_nn" in fileset else None
                        ),
                        "running_container": (
                            data["acc_nn_running"]
                            if "acc_nn_running" in fileset
                            else None
                        ),
                        "loss_container": (
                            data["acc_nn_loss"] if "acc_nn_loss" in fileset else None
                        ),
                    }

                run_key = (
                    meta["power"],
                    meta["thresholded"],
                    meta["stdz"],
                    meta["train_mode"],
                    meta["num_frames"],
                    meta["seed"],
                )

                raw_runs[model_kind][run_key] = model_record
                scan_counts[f"{model_kind}_loaded"] += 1

                if "acc_linear" in fileset:
                    linear_record = {
                        **base_record,
                        "model_kind": "linear",
                        "from_model_source": source_name,
                        "final_container": data["acc_linear"],
                        "running_container": None,
                        "loss_container": None,
                    }

                    existing = raw_runs["linear"].get(run_key)
                    if existing is None:
                        raw_runs["linear"][run_key] = linear_record
                        scan_counts["linear_loaded"] += 1
                    else:
                        old_rank = preferred_source_rank(
                            existing.get(
                                "from_model_source", existing.get("source", "")
                            )
                        )
                        new_rank = preferred_source_rank(source_name)
                        if new_rank < old_rank:
                            raw_runs["linear"][run_key] = linear_record

        except Exception as e:
            warn(f"Failed to load {path}: {e}")

# ====================
# Per-run selection for curve_vs_K
# ====================


def infer_missing_K_list(record):
    K_list = normalize_K_list(record.get("K_list", []))
    if K_list.size > 0:
        return K_list

    nK = max(
        seq_len(record.get("final_container")),
        seq_len(record.get("running_container")),
        seq_len(record.get("loss_container")),
    )
    if nK <= 0:
        return np.array([], dtype=int)

    warn(f"Inferred K_list as 1..{nK} for record {record.get('path', '<unknown>')}")
    return np.arange(1, nK + 1, dtype=int)


def summarize_single_run_vs_K(record, model_kind):
    K_list = infer_missing_K_list(record)
    nK = len(K_list)

    selected_epoch = np.full(nK, -1, dtype=int)
    selected_train_acc = np.full(nK, np.nan, dtype=float)
    selected_val_acc = np.full(nK, np.nan, dtype=float)
    selected_test_acc = np.full(nK, np.nan, dtype=float)
    selected_val_loss = np.full(nK, np.nan, dtype=float)
    selected_test_loss = np.full(nK, np.nan, dtype=float)

    for iK, K in enumerate(K_list):
        if model_kind == "linear":
            final_entry = get_k_entry(
                record.get("final_container"), iK, name=f"linear_final@K={K}"
            )
            train_acc, val_acc, test_acc = parse_final_triplet(
                final_entry, name=f"linear_final@K={K}"
            )
            selected_epoch[iK] = -1
            selected_train_acc[iK] = train_acc
            selected_val_acc[iK] = val_acc
            selected_test_acc[iK] = test_acc
            selected_val_loss[iK] = np.nan
            selected_test_loss[iK] = np.nan
            continue

        running_entry = get_k_entry(
            record.get("running_container"), iK, name=f"{model_kind}_running@K={K}"
        )
        train_hist, val_hist, test_hist = parse_running_triplet(
            running_entry, name=f"{model_kind}_running@K={K}"
        )

        loss_entry = get_k_entry(
            record.get("loss_container"), iK, name=f"{model_kind}_loss@K={K}"
        )
        train_loss_hist, val_loss_hist, test_loss_hist = parse_running_triplet(
            loss_entry, name=f"{model_kind}_loss@K={K}"
        )

        best_epoch = select_best_epoch_from_val_acc(
            val_hist,
            train_hist=train_hist,
            test_hist=test_hist,
            val_loss_hist=val_loss_hist,
            test_loss_hist=test_loss_hist,
        )

        if best_epoch >= 0:
            selected_epoch[iK] = best_epoch
            selected_train_acc[iK] = safe_hist_value(
                train_hist, best_epoch, name=f"train_hist@K={K}"
            )
            selected_val_acc[iK] = safe_hist_value(
                val_hist, best_epoch, name=f"val_hist@K={K}"
            )
            selected_test_acc[iK] = safe_hist_value(
                test_hist, best_epoch, name=f"test_hist@K={K}"
            )
            selected_val_loss[iK] = safe_hist_value(
                val_loss_hist, best_epoch, name=f"val_loss_hist@K={K}"
            )
            selected_test_loss[iK] = safe_hist_value(
                test_loss_hist, best_epoch, name=f"test_loss_hist@K={K}"
            )
        else:
            # Fallback: if running histories are unavailable, use final tuple directly.
            final_entry = get_k_entry(
                record.get("final_container"), iK, name=f"{model_kind}_final@K={K}"
            )
            train_acc, val_acc, test_acc = parse_final_triplet(
                final_entry, name=f"{model_kind}_final@K={K}"
            )
            selected_epoch[iK] = -1
            selected_train_acc[iK] = train_acc
            selected_val_acc[iK] = val_acc
            selected_test_acc[iK] = test_acc
            selected_val_loss[iK] = np.nan
            selected_test_loss[iK] = np.nan

    return {
        "K_list": np.asarray(K_list).copy(),
        "selected_epoch_per_K": selected_epoch,
        "selected_train_acc_per_K": selected_train_acc,
        "selected_val_acc_per_K": selected_val_acc,
        "selected_test_acc_per_K": selected_test_acc,
        "selected_val_loss_per_K": selected_val_loss,
        "selected_test_loss_per_K": selected_test_loss,
    }


def build_curve_vs_K(raw_runs_model, model_kind):
    grouped = defaultdict(dict)
    for run_key, record in raw_runs_model.items():
        power, thresholded, stdz, train_mode, num_frames, seed = run_key
        grouped[(power, thresholded, stdz, train_mode, num_frames)][seed] = record

    curve_vs_K = {}

    for key5, seed_to_record in sorted(grouped.items(), key=lambda kv: kv[0]):
        actual_seed_list = sorted(seed_to_record.keys())

        all_K = []
        per_seed_compact = {}
        for seed in actual_seed_list:
            compact = summarize_single_run_vs_K(
                seed_to_record[seed], model_kind=model_kind
            )
            per_seed_compact[seed] = compact
            if compact["K_list"].size > 0:
                all_K.append(np.asarray(compact["K_list"]).reshape(-1))

        if len(all_K) == 0:
            warn(f"No K_list found for {model_kind} setting {key5}")
            continue

        K_union = np.unique(np.concatenate(all_K))
        if np.allclose(K_union, np.round(K_union), atol=1e-12, rtol=0):
            K_union = np.round(K_union).astype(int)

        n_seed = len(actual_seed_list)
        n_K = len(K_union)

        selected_epoch_per_seed = np.full((n_seed, n_K), -1, dtype=int)
        selected_train_acc_per_seed = np.full((n_seed, n_K), np.nan, dtype=float)
        selected_val_acc_per_seed = np.full((n_seed, n_K), np.nan, dtype=float)
        selected_test_acc_per_seed = np.full((n_seed, n_K), np.nan, dtype=float)
        selected_val_loss_per_seed = np.full((n_seed, n_K), np.nan, dtype=float)
        selected_test_loss_per_seed = np.full((n_seed, n_K), np.nan, dtype=float)

        per_seed = {}

        for i_seed, seed in enumerate(actual_seed_list):
            compact = per_seed_compact[seed]
            run_K = compact["K_list"]
            k_to_idx = {}
            for idx_local, K in enumerate(run_K):
                if K not in k_to_idx:
                    k_to_idx[K] = idx_local

            for iK_global, K in enumerate(K_union):
                if K not in k_to_idx:
                    continue
                idx_local = k_to_idx[K]
                selected_epoch_per_seed[i_seed, iK_global] = int(
                    compact["selected_epoch_per_K"][idx_local]
                )
                selected_train_acc_per_seed[i_seed, iK_global] = compact[
                    "selected_train_acc_per_K"
                ][idx_local]
                selected_val_acc_per_seed[i_seed, iK_global] = compact[
                    "selected_val_acc_per_K"
                ][idx_local]
                selected_test_acc_per_seed[i_seed, iK_global] = compact[
                    "selected_test_acc_per_K"
                ][idx_local]
                selected_val_loss_per_seed[i_seed, iK_global] = compact[
                    "selected_val_loss_per_K"
                ][idx_local]
                selected_test_loss_per_seed[i_seed, iK_global] = compact[
                    "selected_test_loss_per_K"
                ][idx_local]

            per_seed[seed] = {
                "K_list": np.asarray(K_union).copy(),
                "selected_epoch_per_K": selected_epoch_per_seed[i_seed].copy(),
                "selected_train_acc_per_K": selected_train_acc_per_seed[i_seed].copy(),
                "selected_val_acc_per_K": selected_val_acc_per_seed[i_seed].copy(),
                "selected_test_acc_per_K": selected_test_acc_per_seed[i_seed].copy(),
                "selected_val_loss_per_K": selected_val_loss_per_seed[i_seed].copy(),
                "selected_test_loss_per_K": selected_test_loss_per_seed[i_seed].copy(),
            }

        curve_vs_K[key5] = {
            "K_list": np.asarray(K_union).copy(),
            "seed_list": actual_seed_list,
            "selected_epoch_per_seed": selected_epoch_per_seed,
            "selected_val_acc_per_seed": selected_val_acc_per_seed,
            "selected_test_acc_per_seed": selected_test_acc_per_seed,
            "selected_train_acc_per_seed": selected_train_acc_per_seed,
            "selected_val_loss_per_seed": selected_val_loss_per_seed,
            "selected_test_loss_per_seed": selected_test_loss_per_seed,
            "mean_val_acc": nanmean_safe(selected_val_acc_per_seed, axis=0),
            "std_val_acc": nanstd_safe(selected_val_acc_per_seed, axis=0),
            "mean_test_acc": nanmean_safe(selected_test_acc_per_seed, axis=0),
            "std_test_acc": nanstd_safe(selected_test_acc_per_seed, axis=0),
            "per_seed": per_seed,
        }

    return curve_vs_K


# ====================
# curve_vs_shots
# ====================


def build_curve_vs_shots(curve_vs_K, model_kind):
    grouped = defaultdict(dict)
    for key5, summaryK in curve_vs_K.items():
        power, thresholded, stdz, train_mode, num_frames = key5
        grouped[(power, thresholded, stdz, train_mode)][num_frames] = summaryK

    curve_vs_shots = {}

    for key4, shot_to_summary in sorted(grouped.items(), key=lambda kv: kv[0]):
        num_frames_sorted = np.array(sorted(shot_to_summary.keys()), dtype=int)

        seed_union = sorted(
            {
                seed
                for summaryK in shot_to_summary.values()
                for seed in summaryK["seed_list"]
            }
        )

        n_seed = len(seed_union)
        n_shot = len(num_frames_sorted)

        selected_K_per_seed = np.full((n_seed, n_shot), np.nan, dtype=float)
        selected_epoch_per_seed = np.full((n_seed, n_shot), -1, dtype=int)
        selected_val_acc_per_seed = np.full((n_seed, n_shot), np.nan, dtype=float)
        selected_test_acc_per_seed = np.full((n_seed, n_shot), np.nan, dtype=float)
        selected_val_loss_per_seed = np.full((n_seed, n_shot), np.nan, dtype=float)
        selected_test_loss_per_seed = np.full((n_seed, n_shot), np.nan, dtype=float)

        per_seed = {}

        for i_seed, seed in enumerate(seed_union):
            for i_shot, num_frames in enumerate(num_frames_sorted):
                summaryK = shot_to_summary.get(int(num_frames))
                if summaryK is None:
                    continue

                seed_detail = summaryK["per_seed"].get(seed)
                if seed_detail is None:
                    continue

                K_list = np.asarray(seed_detail["K_list"])
                epoch_per_K = np.asarray(seed_detail["selected_epoch_per_K"])
                val_acc_per_K = np.asarray(seed_detail["selected_val_acc_per_K"])
                test_acc_per_K = np.asarray(seed_detail["selected_test_acc_per_K"])
                val_loss_per_K = np.asarray(seed_detail["selected_val_loss_per_K"])
                test_loss_per_K = np.asarray(seed_detail["selected_test_loss_per_K"])

                best_idx = select_best_K_from_val_acc(
                    K_array=K_list,
                    epoch_array=epoch_per_K,
                    val_acc_array=val_acc_per_K,
                )

                if best_idx < 0:
                    continue

                selected_K_per_seed[i_seed, i_shot] = float(K_list[best_idx])
                selected_epoch_per_seed[i_seed, i_shot] = int(epoch_per_K[best_idx])
                selected_val_acc_per_seed[i_seed, i_shot] = float(
                    val_acc_per_K[best_idx]
                )
                selected_test_acc_per_seed[i_seed, i_shot] = float(
                    test_acc_per_K[best_idx]
                )
                selected_val_loss_per_seed[i_seed, i_shot] = (
                    float(val_loss_per_K[best_idx])
                    if np.isfinite(val_loss_per_K[best_idx])
                    else np.nan
                )
                selected_test_loss_per_seed[i_seed, i_shot] = (
                    float(test_loss_per_K[best_idx])
                    if np.isfinite(test_loss_per_K[best_idx])
                    else np.nan
                )

            per_seed[seed] = {
                "num_frames_list": num_frames_sorted.copy(),
                "selected_K_per_shot": selected_K_per_seed[i_seed].copy(),
                "selected_epoch_per_shot": selected_epoch_per_seed[i_seed].copy(),
                "selected_val_acc_per_shot": selected_val_acc_per_seed[i_seed].copy(),
                "selected_test_acc_per_shot": selected_test_acc_per_seed[i_seed].copy(),
                "selected_val_loss_per_shot": selected_val_loss_per_seed[i_seed].copy(),
                "selected_test_loss_per_shot": selected_test_loss_per_seed[
                    i_seed
                ].copy(),
            }

        curve_vs_shots[key4] = {
            "num_frames_list": num_frames_sorted.copy(),
            "seed_list": seed_union,
            "selected_K_per_seed": selected_K_per_seed,
            "selected_epoch_per_seed": selected_epoch_per_seed,
            "selected_val_acc_per_seed": selected_val_acc_per_seed,
            "selected_test_acc_per_seed": selected_test_acc_per_seed,
            "selected_val_loss_per_seed": selected_val_loss_per_seed,
            "selected_test_loss_per_seed": selected_test_loss_per_seed,
            "mean_test_acc": nanmean_safe(selected_test_acc_per_seed, axis=0),
            "std_test_acc": nanstd_safe(selected_test_acc_per_seed, axis=0),
            "mean_val_acc": nanmean_safe(selected_val_acc_per_seed, axis=0),
            "std_val_acc": nanstd_safe(selected_val_acc_per_seed, axis=0),
            "per_seed": per_seed,
        }

    return curve_vs_shots


# ====================
# Build final summary
# ====================

results_summary = {
    "logistic": {},
    "nn": {},
    "linear": {},
}

for model_kind in ["logistic", "nn", "linear"]:
    curve_vs_K = build_curve_vs_K(raw_runs[model_kind], model_kind=model_kind)
    curve_vs_shots = build_curve_vs_shots(curve_vs_K, model_kind=model_kind)
    results_summary[model_kind]["curve_vs_K"] = curve_vs_K
    results_summary[model_kind]["curve_vs_shots"] = curve_vs_shots

# ====================
# Compact printout
# ====================

print("=== raw_runs summary ===")
print(f"logistic runs loaded: {len(raw_runs['logistic'])}")
print(f"nn runs loaded      : {len(raw_runs['nn'])}")
print(f"linear runs loaded  : {len(raw_runs['linear'])}")
print(f"scan counts         : {scan_counts}")

print("\n=== results_summary overview ===")
for model_kind in ["logistic", "nn", "linear"]:
    curve_vs_K = results_summary[model_kind]["curve_vs_K"]
    curve_vs_shots = results_summary[model_kind]["curve_vs_shots"]
    print(
        f"{model_kind:8s} | curve_vs_K settings = {len(curve_vs_K):4d} | curve_vs_shots settings = {len(curve_vs_shots):4d}"
    )

    if len(curve_vs_K) > 0:
        ex_key_K = next(iter(curve_vs_K))
        ex_sum_K = curve_vs_K[ex_key_K]
        print(f"  example curve_vs_K key      : {ex_key_K}")
        print(f"  K_list shape                : {ex_sum_K['K_list'].shape}")
        print(
            f"  selected_epoch_per_seed     : {ex_sum_K['selected_epoch_per_seed'].shape}"
        )
        print(
            f"  selected_test_acc_per_seed  : {ex_sum_K['selected_test_acc_per_seed'].shape}"
        )

    if len(curve_vs_shots) > 0:
        ex_key_S = next(iter(curve_vs_shots))
        ex_sum_S = curve_vs_shots[ex_key_S]
        print(f"  example curve_vs_shots key  : {ex_key_S}")
        print(f"  num_frames_list shape       : {ex_sum_S['num_frames_list'].shape}")
        print(
            f"  selected_K_per_seed         : {ex_sum_S['selected_K_per_seed'].shape}"
        )
        print(
            f"  selected_test_acc_per_seed  : {ex_sum_S['selected_test_acc_per_seed'].shape}"
        )

## Figure 2 (MNIST) & Supplementary MNIST High Power

In [ ]:
fig_dir = {
    "low": fig_folder_dir / "fig2",
    "high": fig_folder_dir / "supp_MNISTHighPower",
}

for power in power_list:
    fig_dir[power].mkdir(parents=True, exist_ok=True)

none_stdz_plot = [
    "eigen",
    # "pca",
    # "cg",
    # "lpfft",
]  # train_modes in this list use stdz=False; others use stdz=True
fitting_name = {"nn": "NN", "logistic": "Logistic"}

### Accuracy vs Features

In [ ]:
thresholded = False
mark_x = np.array([2, 4, 9, 16, 25, 100], dtype=float)

for fitting_plot in ["logistic", "nn"]:
    model_kind = fitting_plot
    for power in power_list:
        num_frames = 10 if power == "low" else 5
        scale = 1
        fig, ax = plt.subplots(1, 1, figsize=(2.4 * scale, 1.9 * scale))

        series = {}
        for train_mode in train_mode_list:
            stdz = (
                False
                if (train_mode in none_stdz_plot and fitting_plot == "logistic")
                else True
            )

            key = (power, thresholded, stdz, train_mode, num_frames)
            if key not in results_summary[model_kind]["curve_vs_K"]:
                print(f"[warning] missing {model_kind} key: {key}")
                continue

            summary = results_summary[model_kind]["curve_vs_K"][key]
            x = np.asarray(summary["K_list"], dtype=float)
            y = np.asarray(summary["mean_test_acc"], dtype=float)
            yerr = np.asarray(summary["std_test_acc"], dtype=float)

            valid = np.isfinite(x) & np.isfinite(y)
            x = x[valid]
            y = y[valid]
            yerr = yerr[valid]

            order = np.argsort(x)
            x = x[order]
            y = y[order]
            yerr = yerr[order]

            series[train_mode] = {"x": x, "y": y, "yerr": yerr}

            ax.errorbar(
                x,
                y,
                yerr,
                marker="o",
                markerfacecolor=color_dic[train_mode] + [face_alpha],
                markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
                color=color_dic[train_mode] + [line_alpha],
                label=legend_dic[train_mode],
            )

        if fitting_plot == "logistic" and power == "low":
            ax.set_ylim(23, 70)
        elif fitting_plot == "nn" and power == "low":
            ax.set_ylim(23, 75)
        elif fitting_plot == "logistic" and power == "high":
            ax.set_ylim(60, 90)
        elif fitting_plot == "nn" and power == "high":
            ax.set_ylim(60, 95)

        ax.vlines(
            9, *ax.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
        )
        ax.vlines(
            25, *ax.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
        )

        ax.set_title(f"{fitting_name[fitting_plot]}, {num_frames} shots")
        ax.set_ylabel("Test accuracy (%)")
        ax.set_xlabel(r"Number of reduced features $K_{\mathrm{r}}$")
        ax.set_xscale("log")
        ax.set_xticks(np.array([2, 3, 4, 5, 10, 20]) ** 2)
        ax.set_xticklabels(np.array([2, 3, 4, 5, 10, 20]) ** 2)
        ax.set_xlim(1, 400)
        ax.grid(alpha=0.9, which="major", linestyle="--")

        adjust_fixed_margins(
            fig, left_in=0.45, bottom_in=0.35, right_in=0.10, top_in=0.18
        )

        fig.savefig(
            fig_dir[power]
            / f"acc_feature_{power}_power_{num_frames}_shots_{fitting_plot}.svg",
            format="svg",
            dpi=600,
            bbox_inches=None,
            pad_inches=0.0,
        )
        plt.show()

### Accuracy vs Shots

In [ ]:
for fitting_plot in ["logistic", "nn"]:
    model_kind = fitting_plot
    for power in power_list:
        scale = 1
        fig, ax = plt.subplots(1, 1, figsize=(2.4 * scale, 1.9 * scale))

        for train_mode in train_mode_list:
            stdz = (
                False
                if (train_mode in none_stdz_plot and fitting_plot == "logistic")
                else True
            )

            key = (power, thresholded, stdz, train_mode)
            if key not in results_summary[model_kind]["curve_vs_shots"]:
                print(f"[warning] missing {model_kind} key: {key}")
                continue

            summary = results_summary[model_kind]["curve_vs_shots"][key]
            x = np.asarray(summary["num_frames_list"], dtype=float)
            y = np.asarray(summary["mean_test_acc"], dtype=float)
            yerr = np.asarray(summary["std_test_acc"], dtype=float)

            valid = np.isfinite(x) & np.isfinite(y)
            x = x[valid]
            y = y[valid]
            yerr = yerr[valid]

            order = np.argsort(x)
            x = x[order]
            y = y[order]
            yerr = yerr[order]

            ax.errorbar(
                x,
                y,
                yerr,
                markerfacecolor=color_dic[train_mode] + [face_alpha],
                markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
                color=color_dic[train_mode] + [line_alpha],
                label=legend_dic[train_mode],
            )

        ax.vlines(
            (10 if power == "low" else 5),
            *ax.get_ylim(),
            colors="0.3",
            linestyles="--",
            linewidth=0.5,
            zorder=0,
        )
        ax.set_title(f"{fitting_name[fitting_plot]}")
        ax.set_ylabel("Test accuracy (%)")
        ax.set_xlabel(r"Number of shots $S$")
        ax.set_xticks(range(0, 51, 10))
        ax.set_xlim(0, 50)

        # if power == "high":
        #     ax.set_yticks(range(70, 91, 5))
        if fitting_plot == "logistic" and power == "low":
            ax.set_ylim(23, 70)
        elif fitting_plot == "nn" and power == "low":
            ax.set_ylim(23, 75)
        elif fitting_plot == "logistic" and power == "high":
            ax.set_ylim(60, 90)
        elif fitting_plot == "nn" and power == "high":
            ax.set_ylim(60, 95)

        ax.grid(alpha=0.9, which="major", linestyle="--")

        adjust_fixed_margins(
            fig, left_in=0.45, bottom_in=0.35, right_in=0.10, top_in=0.18
        )

        fig.savefig(
            fig_dir[power] / f"acc_shot_{power}_power_{fitting_plot}.svg",
            format="svg",
            dpi=600,
            bbox_inches=None,
            pad_inches=0.0,
        )
        plt.show()

### SNR

In [ ]:
NTrain, NVal, NTest = 8000, 2000, 2000
N_samp = NTrain + NTest + NVal
pixels = 45
K_max = pixels**2
max_frame = num_frames = 100
seed = 0
V_from_max_frame = True
thresholded = False

V_frame = max_frame if V_from_max_frame else num_frames

r_train_record = {}
pca_weight_record = {}
for power in power_list:
    np.random.seed(seed)
    perm = np.random.permutation(N_samp)
    labels = np.load(MNIST_LENS_DIR[power] / "mnist_metadata.npz")["labels"]
    count = {i: 0 for i in range(10)}
    train_indices, val_indices, test_indices = [], [], []
    for i in range(N_samp):
        count[labels[perm[i]]] += 1
        if count[labels[perm[i]]] <= NTrain / 10:
            train_indices.append(perm[i])
        elif count[labels[perm[i]]] <= (NTrain + NTest) / 10:
            test_indices.append(perm[i])
        else:
            val_indices.append(perm[i])
    perm = np.array(test_indices + val_indices + train_indices)
    data_labels = labels[perm]

    data_images = []
    data_train = []
    V = np.zeros((K_max, K_max))
    for i in tqdm(range(N_samp)):
        frames = np.load(dataset_dir[power] / f"{perm[i]}_EM_Gain_200.npz")["data"]
        data_images.append(np.mean(frames[:num_frames], axis=0))
        XMat = frames[:V_frame].reshape(V_frame, -1).T
        if i >= NTest + NVal:
            # V += np.cov(np.apply_along_axis(lambda row: np.random.shuffle(row) or row, axis=1, arr=XMat.copy())) # for eishuffle only
            V += np.cov(XMat)
            data_train.append(np.mean(frames, axis=0))
    V = V / NTrain

    V_norm = np.zeros((K_max, K_max))
    for i in tqdm(range(len(V))):
        for j in range(i + 1):
            V_norm[i, j] = V[i, j] / np.sqrt(V[i, i] * V[j, j])
            V_norm[j, i] = V[j, i] / np.sqrt(V[i, i] * V[j, j])

    data_images = np.array(data_images)
    data_train = np.array(data_train).reshape(len(data_train), -1)
    data_labels = np.load(MNIST_LENS_DIR[power] / "mnist_metadata.npz")["labels"][perm]
    print("V and averaged data obtained!")

    def standardize_data(data, train_indices=None):
        if train_indices is None:
            train_indices = np.arange(len(data))
        train_mean = data[train_indices].mean(axis=0)
        train_std = np.std(data[train_indices], axis=0)
        train_std[train_std == 0] = 1
        return (data - train_mean) / train_std, train_mean, train_std

    # raw data
    data_orig = data_images.reshape(len(data_images), -1)
    data_orig_norm = standardize_data(
        data_orig, train_indices=np.arange(NTest + NVal, N_samp)
    )

    # eigentasks
    _, _, nsr, r_train = eigentask_solver(data_train, V=V)
    data_eigen = data_orig @ r_train.T
    r_train_record[power] = r_train
    data_eigen_norm, eigen_mean, eigen_std = standardize_data(
        data_eigen, train_indices=np.arange(NTest + NVal, N_samp)
    )

    # low pass filtering
    data_lpfft = fft(data_images)
    data_lpfft_norm, _, _ = standardize_data(
        data_lpfft, train_indices=np.arange(NTest + NVal, N_samp)
    )

    # pca
    _, pca = pca_solver(data_train.reshape(NTrain, -1))
    data_pca = (
        data_orig
        - np.tile(np.mean(data_train.reshape(NTrain, -1), axis=0), (N_samp, 1))
    ) @ (pca).T
    data_pca_norm, pca_mean, pca_std = standardize_data(
        data_pca, train_indices=np.arange(NTest + NVal, N_samp)
    )
    pca_weight_record[power] = pca

    eg_snr_test = []
    pca_snr_test = []
    eg_std_test = []
    pca_std_test = []
    for i in tqdm(test_indices):
        frames = np.load(dataset_dir[power] / f"{i}_EM_Gain_200.npz")["data"]
        frames = frames.reshape(100, -1)
        #
        EG_data_single_MNIST = ((frames) @ r_train.T - eigen_mean) / eigen_std
        PCA_data_single_MNIST = (
            (frames - data_train.reshape(NTrain, -1).mean(axis=0)) @ (pca).T - pca_mean
        ) / pca_std
        #
        eg_snr_test.append(
            np.abs(
                np.mean(EG_data_single_MNIST, axis=0)
                / np.std(EG_data_single_MNIST, axis=0)
            )
        )
        pca_snr_test.append(
            np.abs(
                np.mean(PCA_data_single_MNIST, axis=0)
                / np.std(PCA_data_single_MNIST, axis=0)
            )
        )
        #
        eg_std_test.append(np.std(EG_data_single_MNIST, axis=0))
        pca_std_test.append(np.std(PCA_data_single_MNIST, axis=0))

    eg_snr_test = np.array(eg_snr_test)
    pca_snr_test = np.array(pca_snr_test)
    eg_std_test = np.array(eg_std_test)
    pca_std_test = np.array(pca_std_test)

    for N_features in [9, 25]:
        # PCA
        PCA_face = [254, 207, 146, 100]
        # Eigentasks
        Eig_face = [183, 55, 121, 255]

        fig, ax = plt.subplots(1, 1, figsize=(1.9, 1.9))
        ax.hist(
            eg_snr_test[0:2000, 1 : (N_features + 1)].flatten(),
            bins=50,
            range=((0.0, 5) if power == "high" else (0, 2.5)),
            color=np.array(Eig_face) / 255,
            label="Eigentasks",
            alpha=0.8,
        )
        ax.hist(
            pca_snr_test[0:2000, :N_features].flatten(),
            bins=50,
            range=((0.0, 5) if power == "high" else (0, 2.5)),
            color=np.array(PCA_face) / 255,
            label="PCA components",
            alpha=0.8,
        )
        ax.set_yscale("log")
        ax.set_title(f"First {N_features} features")
        ax.set_xlabel(r"Estimate of SNR")
        ax.set_ylabel(r"Frequency of features")
        if power == "high":
            ax.set_xticks(range(0, 6, 1))
        else:
            ax.set_xticks(np.arange(0, 2.5, 0.5))
        ax.legend()

        adjust_fixed_margins(
            fig, left_in=0.45, bottom_in=0.35, right_in=0.10, top_in=0.18
        )
        fig.savefig(
            fig_dir[power]
            / f"Eig_PCA_feature_SNR_{power}_power_MNIST_first{N_features}.svg",
            format="svg",
            dpi=600,
            bbox_inches=None,
            pad_inches=0.0,
        )
        plt.show()

#### Eigentask Visualization

In [ ]:
for power in ["low"]:
    scale = 1
    width = 6.6 - 0.44

    idx_list = np.array([1, 2, 3, 4, 9, 16, 25, 100, 2025]) - 1
    n_row, n_col = 1, len(idx_list)

    fig, ax = plt.subplots(
        n_row, n_col, dpi=150, figsize=(width, width / (1.1 * n_col))
    )

    fig.patch.set_facecolor("none")
    for a in ax:
        a.set_facecolor("none")

    imgs = [r_train_record[power][i].reshape(45, 45) for i in idx_list]
    vmin = min(im.min() for im in imgs)
    vmax = max(im.max() for im in imgs)

    im = None
    for j, (idx, img) in enumerate(zip(idx_list, imgs)):
        im = ax[j].imshow(img, cmap="magma", vmin=vmin, vmax=vmax)
        ax[j].text(
            2,
            3,
            rf"$r^{{({idx+1})}}$",
            fontsize=7,
            color="white",
            ha="left",
            va="top",
        )
        ax[j].set_xticks([])
        ax[j].set_yticks([])
        ax[j].set_frame_on(False)

    fig.subplots_adjust(wspace=0.1, hspace=0)

    adjust_fixed_margins(fig, left_in=0, bottom_in=0, right_in=0, top_in=0)
    fig.savefig(
        fig_dir[power] / f"eigen_weights_{power}.svg",
        format="svg",
        dpi=600,
        transparent=True,
        bbox_inches=None,
        pad_inches=0.0,
    )

## Supplementary Eigentask Visualization

In [ ]:
# Need to run the above cells to get r_train

fig_dir = fig_folder_dir / "supp_MNISTEigenVis"

for power in power_list:
    fig_dir.mkdir(parents=True, exist_ok=True)

for power in power_list:
    scale = 1
    width = 6.7

    idx_1based_list = [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9,
        10,
        11,
        12,
        13,
        14,
        15,
        16,
        17,
        18,
        19,
        20,
        21,
        22,
        23,
        24,
        25,
        26,
        27,
        28,
        29,
        30,
        31,
        32,
        33,
        34,
        35,
        36,
        37,
        38,
        39,
        40,
        41,
        42,
        43,
        44,
        45,
        46,
        47,
        48,
        49,
        50,
        64,
        81,
        100,
        225,
        400,
        625,
        900,
        1225,
        1600,
        2025,
    ]
    if len(idx_1based_list) != 60:
        raise ValueError(
            f"idx_1based_list must have length 60, got {len(idx_1based_list)}"
        )

    idx_list = np.array(idx_1based_list, dtype=int) - 1  # 0-based

    n_row, n_col = 6, 10
    if len(idx_list) != n_row * n_col:
        raise ValueError("idx_list length must be n_row*n_col (60).")

    # Figure size (keep panels roughly square)
    fig_w = width
    fig_h = width / (1.1 * n_col) * n_row

    # Use GridSpec: last column is a dedicated colorbar axis spanning all rows
    fig = plt.figure(dpi=150, figsize=(fig_w, fig_h))
    gs = fig.add_gridspec(
        n_row,
        n_col + 1,
        width_ratios=[1] * n_col + [0.2],  # last column reserved for colorbar
        wspace=0.1,
        hspace=0.1,
    )

    ax = np.empty((n_row, n_col), dtype=object)
    for r in range(n_row):
        for c in range(n_col):
            ax[r, c] = fig.add_subplot(gs[r, c])
    cax = fig.add_subplot(gs[1:-1, -1])  # spans 6 rows

    # --- transparent background (Figure + Axes) ---
    fig.patch.set_facecolor("none")
    for a in ax.ravel():
        a.set_facecolor("none")
    cax.set_facecolor("none")

    # --- shared color scale ---
    record = r_train_record[power]
    max_needed = int(idx_list.max())
    if max_needed >= len(record):
        raise IndexError(
            f"Need r_train_record['{power}'] length > {max_needed}, got {len(record)}"
        )

    imgs = [record[i].reshape(45, 45) for i in idx_list]
    vmin = min(img.min() for img in imgs)
    vmax = max(img.max() for img in imgs)

    # --- plot images ---
    im = None
    for j, (idx0, img) in enumerate(zip(idx_list, imgs)):
        r, c = divmod(j, n_col)
        im = ax[r, c].imshow(img, cmap="magma", vmin=vmin, vmax=vmax)
        ax[r, c].text(
            2,
            3,
            rf"$r^{{({idx0+1})}}$",
            fontsize=7,
            color="white",
            ha="left",
            va="top",
        )
        ax[r, c].set_xticks([])
        ax[r, c].set_yticks([])
        ax[r, c].set_frame_on(False)

    # --- colorbar on the right, spanning all 6 rows ---
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.set_facecolor("none")

    adjust_fixed_margins(fig, left_in=0, bottom_in=0, right_in=0.5, top_in=0)
    fig.savefig(
        fig_dir / f"eigen_weights_{power}.svg",
        format="svg",
        dpi=600,
        transparent=True,
        bbox_inches=None,
        pad_inches=0.0,
    )

## Supplementary MNIST Digit and Feature Visualization

In [ ]:
fig_dir = fig_folder_dir / "supp_MNISTFeatureVis"

for power in power_list:
    fig_dir.mkdir(parents=True, exist_ok=True)

none_stdz_plot = [
    "eigen",
    # "pca",
    # "cg",
    # "lpfft",
]  # train_modes in this list use stdz=False; others use stdz=True
fitting_name = {"nn": "NN", "logistic": "Logistic"}

n_low_frames = 2
n_high_frames = 100
digits = list(range(10))
thresholded = False

out_name = "supp_MNISTFeatureVis.svg"


def adjust_fixed_margins(fig, left_in, bottom_in, right_in, top_in):
    w_in, h_in = fig.get_size_inches()
    fig.subplots_adjust(
        left=left_in / w_in,
        right=1 - right_in / w_in,
        bottom=bottom_in / h_in,
        top=1 - top_in / h_in,
    )


def load_avg_frame(dataset_dir, sample_id: int, n_frames: int):
    frames = np.load(dataset_dir / f"{sample_id}_EM_Gain_200.npz")["data"].astype(
        np.float32
    )
    frames = frames[:n_frames]

    avg = frames.mean(axis=0)
    if avg.shape != (pixels, pixels):
        avg = avg.reshape(pixels, pixels)
    return avg


# -------------------- load mnist_metadata.npz (images for row 1) --------------------
train_npz = np.load(MNIST_LENS_DIR["high"] / "mnist_metadata.npz")
labels = train_npz["labels"]
images = train_npz["images"]  # (12000, 28, 28)

# pick one sample per digit (first occurrence)
sample_id_per_digit = {}
for d in digits:
    idxs = np.where(labels == d)[0]
    if len(idxs) == 0:
        raise ValueError(f"Digit {d} not found in labels.")
    sample_id_per_digit[d] = int(idxs[0])

mnist_imgs, raw_imgs = [], {power: [] for power in power_list}
for d in digits:
    sid = sample_id_per_digit[d]
    mnist_imgs.append(images[sid])
    for power in power_list:
        raw_imgs[power].append(load_avg_frame(dataset_dir[power], sid, n_low_frames))

vmin, vmax = {}, {}
for power in power_list:
    vmin[power], vmax[power] = min(im.min() for im in raw_imgs[power]), max(
        im.max() for im in raw_imgs[power]
    )

# -------------------- plot: 3x10 images, colorbars added manually (guaranteed gap) --------------------
n_row, n_col = 3, 10
fig = plt.figure(dpi=150, figsize=(6.7, 2.0))

# IMPORTANT: no extra gridspec column for colorbar; we'll place cbar axes manually in the right margin
gs = fig.add_gridspec(n_row, n_col, wspace=0.1, hspace=-0.1)

ax = np.empty((n_row, n_col), dtype=object)
for r in range(n_row):
    for c in range(n_col):
        ax[r, c] = fig.add_subplot(gs[r, c])

# background
fig.patch.set_facecolor("none")
for a in ax.ravel():
    a.set_facecolor("none")

# row labels
ax[0, 0].set_ylabel("MNIST\ndigits", rotation=90)
ax[1, 0].set_ylabel("Low power\n2 shots", rotation=90)
ax[2, 0].set_ylabel("High power\n100 shots", rotation=90)

# row 1: MNIST (images)
for j, d in enumerate(digits):
    a = ax[0, j]
    a.imshow(mnist_imgs[j], cmap="magma")
    a.set_xticks([])
    a.set_yticks([])
    a.set_frame_on(False)

# row 2-3: collected frames
im = {power: None for power in power_list}
for power in power_list:
    for j in range(n_col):
        a = ax[(1 if power == "low" else 2), j]
        im[power] = a.imshow(
            raw_imgs[power][j], cmap="magma", vmin=vmin[power], vmax=vmax[power]
        )
        a.set_xticks([])
        a.set_yticks([])
        a.set_frame_on(False)

# margins: reserve some right margin for colorbars
adjust_fixed_margins(fig, left_in=0.35, bottom_in=0.08, right_in=0.55, top_in=0.06)

# ---- place colorbars manually in the reserved right margin ----
fig.canvas.draw()  # finalize positions

# right edge of last column (in figure coords)
x_right = max(
    ax[0, -1].get_position().x1,
    ax[1, -1].get_position().x1,
    ax[2, -1].get_position().x1,
)

# row bounding boxes (figure coords)
pos_low_row = ax[1, 0].get_position()
pos_high_row = ax[2, 0].get_position()

# colorbar geometry (figure coords)
pad_x = 0.012  # gap between images and colorbar
cbar_w = 0.012  # colorbar width
shrink = 0.18  # shrink inside each row (creates label-safe gap)
min_gap = 0.018  # extra guaranteed gap between the two colorbar axes (figure coords)

x0 = x_right + pad_x

# initial cax positions (inside each row)
y0_low = pos_low_row.y0 + shrink * pos_low_row.height
h_low = (1 - 2 * shrink) * pos_low_row.height
y0_high = pos_high_row.y0 + shrink * pos_high_row.height
h_high = (1 - 2 * shrink) * pos_high_row.height

# enforce a gap between top of high-cbar and bottom of low-cbar
y1_high = y0_high + h_high
if y1_high > y0_low - min_gap:
    overlap = y1_high - (y0_low - min_gap)
    # push low up and high down equally, but keep within their rows
    y0_low += overlap / 2
    y0_high -= overlap / 2
    # clamp back into row bounds if needed
    y0_low = min(y0_low, pos_low_row.y1 - h_low)
    y0_high = max(y0_high, pos_high_row.y0)

cax_low = fig.add_axes([x0, y0_low, cbar_w, h_low])
cax_high = fig.add_axes([x0, y0_high, cbar_w, h_high])
cax_low.set_facecolor("none")
cax_high.set_facecolor("none")

# colorbars
cbar_low = fig.colorbar(im["low"], cax=cax_low)
cbar_high = fig.colorbar(im["high"], cax=cax_high)

# ---- remove numeric ticks; draw "min/max" as text INSIDE each colorbar axis (won't overlap) ----
for cbar in (cbar_low, cbar_high):
    cbar.set_ticks([])
    cbar.ax.tick_params(length=0)
    # place labels in axis coords; x>1 puts text to the right but y stays within the axis box
    cbar.ax.text(
        1.25,
        0.00,
        "min",
        transform=cbar.ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=7,
    )
    cbar.ax.text(
        1.25, 1.00, "max", transform=cbar.ax.transAxes, ha="left", va="top", fontsize=7
    )

fig.savefig(fig_dir / out_name, format="svg", dpi=600, bbox_inches=None, pad_inches=0.0)
plt.show()

## Supplementary MNIST Thresholded Results

In [ ]:
fig_dir = fig_folder_dir / "supp_MNISTThresholding"

fig_dir.mkdir(parents=True, exist_ok=True)

none_stdz_plot = [
    "eigen",
    # "pca",
    # "cg",
    # "lpfft",
]  # train_modes in this list use stdz=False; others use stdz=True
fitting_name = {"nn": "NN", "logistic": "Logistic"}

In [ ]:
thresholded = True

for fitting_plot in ["logistic", "nn"]:
    model_kind = fitting_plot
    for power in ["low"]:
        scale = 1
        fig, ax = plt.subplots(1, 1, figsize=(3.2 * scale, 2.3 * scale))

        # ---------------- main plot: low power ----------------
        for train_mode in train_mode_list:
            stdz = (
                False
                if (train_mode in none_stdz_plot and fitting_plot == "logistic")
                else True
            )

            key = ("low", thresholded, stdz, train_mode)
            if key not in results_summary[model_kind]["curve_vs_shots"]:
                print(f"[warning] missing {model_kind} key: {key}")
                continue

            summary = results_summary[model_kind]["curve_vs_shots"][key]
            x = np.asarray(summary["num_frames_list"], dtype=float)
            y = np.asarray(summary["mean_test_acc"], dtype=float)
            yerr = np.asarray(summary["std_test_acc"], dtype=float)

            valid = np.isfinite(x) & np.isfinite(y)
            x = x[valid]
            y = y[valid]
            yerr = yerr[valid]

            order = np.argsort(x)
            x = x[order]
            y = y[order]
            yerr = yerr[order]

            ax.errorbar(
                x,
                y,
                yerr=yerr,
                markerfacecolor=color_dic[train_mode] + [face_alpha],
                markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
                c=color_dic[train_mode] + [line_alpha],
                label=legend_dic[train_mode],
                zorder=3,
            )

        ax.set_title(f"{fitting_name[fitting_plot]}: thresholded low-power data")
        ax.set_ylabel("Test accuracy (%)")
        ax.set_xlabel(r"Number of shots $S$")
        ax.set_xticks(range(0, 51, 10))
        ax.set_xlim(0, 50)

        ax.set_ylim(23, 65)
        ax.set_yticks(range(30, 70, 10))

        ax.grid(alpha=0.9, which="major", linestyle="--")

        # ---------------- inset: high power ----------------
        inset = fig.add_axes([0.56, 0.25, 0.36, 0.34])

        for train_mode in train_mode_list:
            stdz = (
                False
                if (train_mode in none_stdz_plot and fitting_plot == "logistic")
                else True
            )

            key = ("high", thresholded, stdz, train_mode)
            if key not in results_summary[model_kind]["curve_vs_shots"]:
                print(f"[warning] missing {model_kind} key: {key}")
                continue

            summary = results_summary[model_kind]["curve_vs_shots"][key]
            x = np.asarray(summary["num_frames_list"], dtype=float)
            y = np.asarray(summary["mean_test_acc"], dtype=float)
            yerr = np.asarray(summary["std_test_acc"], dtype=float)

            valid = np.isfinite(x) & np.isfinite(y)
            x = x[valid]
            y = y[valid]
            yerr = yerr[valid]

            order = np.argsort(x)
            x = x[order]
            y = y[order]
            yerr = yerr[order]

            inset.errorbar(
                x,
                y,
                yerr=yerr,
                markersize=1.5 * scale,
                markerfacecolor=color_dic[train_mode] + [face_alpha],
                markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
                c=color_dic[train_mode] + [line_alpha],
                zorder=2,
            )

        inset.set_title("high-power data", fontsize=8, pad=2)
        inset.set_xlim(0, 50)
        inset.set_xticks([0, 10, 20, 30, 40, 50])

        if fitting_plot == "logistic":
            inset.set_ylim(60, 90)
        elif fitting_plot == "nn":
            inset.set_ylim(60, 95)

        inset.grid(alpha=0.9, which="major", linestyle="--")
        inset.tick_params(axis="both", labelsize=7)

        adjust_fixed_margins(
            fig, left_in=0.45, bottom_in=0.35, right_in=0.10, top_in=0.18
        )

        fig.savefig(
            fig_dir / f"acc_shot_low_power_with_high_inset_{fitting_plot}.svg",
            format="svg",
            dpi=600,
            bbox_inches=None,
            pad_inches=0.0,
        )
        plt.show()

# MPEG-7 Figures

## Load Data

In [ ]:
# ====================
# User-editable config
# ====================
logistic_result_dir = RESULT_DIR / "mpeg7_logistic"
nn_result_dir = RESULT_DIR / "mpeg7_nn"
preferred_linear_source = "logistic"  # "logistic" or "nn"

# seed_list = None          # e.g. [0, 1, 2, 3, 4]; None -> infer from files
# N_class_list = None       # e.g. [10, 20, 30, 40, 50, 60, 70]; None -> infer
# num_frames_list = None    # e.g. [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 50]; None -> infer
# stdz_list = None          # e.g. [False, True]; None -> infer
# train_mode_list = None    # e.g. ["eigen", "lpfft", "cg", "pca"]; None -> infer
# thresholded_list = None   # e.g. [False]; None -> infer
# Vmax_list = None          # e.g. [True]; None -> infer

seed_list = range(5)
N_class_list = [70, 60, 50, 40, 30, 20, 10]
stdz_list = [False, True]
num_frames_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 50]
train_mode_list = ["eigen", "lpfft", "cg", "pca"]
Vmax_list = [True]

verbose = True

# ====================
# Helpers
# ====================
FILENAME_RE = re.compile(
    r"n_class_(?P<N_class>\d+)_thresholded_(?P<thresholded>True|False|true|false|0|1)"
    r"_shots_(?P<num_frames>\d+)_Vmax_(?P<Vmax>True|False|true|false|0|1)"
    r"_train_(?P<train_mode>[^_]+)_seed_(?P<seed>-?\d+)_stdz_(?P<stdz>True|False|true|false|0|1)\.npz$"
)


def _warn(msg):
    warnings.warn(str(msg), RuntimeWarning, stacklevel=2)


def parse_bool(x):
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, (int, np.integer)):
        return bool(int(x))
    if isinstance(x, bytes):
        x = x.decode()
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true", "1", "t", "yes", "y"}:
            return True
        if s in {"false", "0", "f", "no", "n"}:
            return False
    raise ValueError(f"Cannot parse bool from {x!r}")


def unwrap_scalar(x):
    if isinstance(x, np.ndarray) and x.shape == ():
        return unwrap_scalar(x.item())
    if isinstance(x, np.generic):
        return x.item()
    return x


def safe_float_scalar(x, default=np.nan):
    try:
        arr = np.asarray(unwrap_scalar(x), dtype=float)
        if arr.size == 0:
            return default
        return float(arr.reshape(-1)[0])
    except Exception:
        return default


def safe_np_load(path):
    with np.load(path, allow_pickle=True) as data:
        out = {k: data[k] for k in data.files}
    out["_path"] = str(path)
    return out


def safe_get(meta, key, default=None):
    if key not in meta:
        return default
    return unwrap_scalar(meta[key])


def to_1d_float_array(x):
    if x is None:
        return np.array([], dtype=float)
    x = unwrap_scalar(x)
    if isinstance(x, np.ndarray) and x.dtype != object:
        return np.asarray(x, dtype=float).reshape(-1)
    if isinstance(x, (list, tuple)):
        try:
            return np.asarray(x, dtype=float).reshape(-1)
        except Exception:
            vals = []
            for item in x:
                vals.extend(to_1d_float_array(item).tolist())
            return np.asarray(vals, dtype=float).reshape(-1)
    if isinstance(x, np.ndarray) and x.dtype == object:
        vals = []
        for item in x.reshape(-1).tolist():
            vals.extend(to_1d_float_array(item).tolist())
        return np.asarray(vals, dtype=float).reshape(-1)
    try:
        return np.asarray([float(x)], dtype=float)
    except Exception:
        return np.array([], dtype=float)


def normalize_k_list(x):
    arr = to_1d_float_array(x)
    if arr.size == 0:
        return np.array([], dtype=int)
    return np.asarray(np.round(arr), dtype=int)


def _object_array_to_nested_list(x):
    if isinstance(x, np.ndarray):
        if x.dtype == object:
            if x.shape == ():
                return _object_array_to_nested_list(x.item())
            return [_object_array_to_nested_list(v) for v in x.tolist()]
        return x
    if isinstance(x, (list, tuple)):
        return [_object_array_to_nested_list(v) for v in x]
    return x


def get_nested_entry(x, indices):
    cur = _object_array_to_nested_list(x)
    try:
        for idx in indices:
            cur = cur[idx]
        return cur
    except Exception:
        return None


def parse_triplet_or_pair(entry, expected_len=None):
    entry = unwrap_scalar(entry)

    if entry is None:
        vals = []

    elif isinstance(entry, np.ndarray):
        if entry.dtype == object:
            nested = _object_array_to_nested_list(entry)
            vals = list(nested) if isinstance(nested, (list, tuple)) else [nested]
        elif entry.ndim == 0:
            vals = [entry.item()]
        else:
            vals = [entry[i] for i in range(entry.shape[0])]

    elif isinstance(entry, (list, tuple)):
        vals = list(entry)

    else:
        vals = [entry]

    if expected_len is not None and len(vals) < expected_len:
        vals = vals + [None] * (expected_len - len(vals))

    return vals


def extract_history(entry, index):
    parts = parse_triplet_or_pair(entry)
    if index >= len(parts):
        return np.array([], dtype=float)
    try:
        return np.asarray(parts[index], dtype=float).reshape(-1)
    except Exception:
        return to_1d_float_array(parts[index])


def extract_scalar_metric(entry, index):
    parts = parse_triplet_or_pair(entry)
    if index >= len(parts):
        return np.nan
    return safe_float_scalar(parts[index], default=np.nan)


def infer_main_keys(npz_dict):
    keys = set(npz_dict.keys())
    if {
        "cv_acc_logistic",
        "cv_acc_logistic_running",
        "full_acc_logistic",
        "full_acc_logistic_running",
    } <= keys:
        return {
            "kind": "logistic",
            "cv_acc_key": "cv_acc_logistic",
            "cv_running_key": "cv_acc_logistic_running",
            "cv_loss_key": "cv_acc_logistic_loss",
            "full_acc_key": "full_acc_logistic",
            "full_running_key": "full_acc_logistic_running",
            "full_loss_key": "full_acc_logistic_loss",
            "linear_cv_key": "cv_acc_linear",
            "linear_full_key": "full_acc_linear",
        }
    if {"cv_acc_nn", "cv_acc_nn_running", "full_acc_nn", "full_acc_nn_running"} <= keys:
        return {
            "kind": "nn",
            "cv_acc_key": "cv_acc_nn",
            "cv_running_key": "cv_acc_nn_running",
            "cv_loss_key": "cv_acc_nn_loss",
            "full_acc_key": "full_acc_nn",
            "full_running_key": "full_acc_nn_running",
            "full_loss_key": "full_acc_nn_loss",
            "linear_cv_key": "cv_acc_linear",
            "linear_full_key": "full_acc_linear",
        }
    return None


def parse_metadata(npz_dict, path):
    m = FILENAME_RE.match(Path(path).name)
    meta = {}
    if m:
        gd = m.groupdict()
        meta["N_class"] = int(gd["N_class"])
        meta["thresholded"] = parse_bool(gd["thresholded"])
        meta["num_frames"] = int(gd["num_frames"])
        meta["Vmax"] = parse_bool(gd["Vmax"])
        meta["train_mode"] = gd["train_mode"]
        meta["seed"] = int(gd["seed"])
        meta["stdz"] = parse_bool(gd["stdz"])
    else:
        meta["N_class"] = None
        meta["thresholded"] = None
        meta["num_frames"] = None
        meta["Vmax"] = None
        meta["train_mode"] = None
        meta["seed"] = None
        meta["stdz"] = None

    field_fallbacks = {
        "seed": ("seed", int),
        "train_mode": ("train_mode", str),
        "num_frames": ("num_frames", int),
        "thresholded": ("thresholded", parse_bool),
        "Vmax": ("Vmax", parse_bool),
    }
    for k, (field_name, cast_fn) in field_fallbacks.items():
        if meta.get(k, None) is None and field_name in npz_dict:
            try:
                meta[k] = cast_fn(safe_get(npz_dict, field_name))
            except Exception:
                pass

    if meta["stdz"] is None:
        s = Path(path).name
        m2 = re.search(r"_stdz_(True|False|true|false|0|1)\.npz$", s)
        if m2:
            meta["stdz"] = parse_bool(m2.group(1))

    if meta["N_class"] is None:
        m2 = re.search(r"n_class_(\d+)", Path(path).name)
        if m2:
            meta["N_class"] = int(m2.group(1))

    return meta


def _mean_surface_from_fold_histories(running_obj, history_index, n_k, path_label=""):
    if running_obj is None:
        return np.full((n_k, 0), np.nan, dtype=float)
    max_epoch = 0
    fold_histories = [[np.array([], dtype=float) for _ in range(n_k)] for _ in range(3)]
    for fold in range(3):
        for k_idx in range(n_k):
            entry = get_nested_entry(running_obj, (fold, k_idx))
            hist = extract_history(entry, history_index)
            fold_histories[fold][k_idx] = hist
            max_epoch = max(max_epoch, hist.size)
    if max_epoch == 0:
        return np.full((n_k, 0), np.nan, dtype=float)

    sum_surface = np.zeros((n_k, max_epoch), dtype=float)
    cnt_surface = np.zeros((n_k, max_epoch), dtype=int)
    for fold in range(3):
        for k_idx in range(n_k):
            hist = fold_histories[fold][k_idx]
            if hist.size == 0:
                continue
            finite = np.isfinite(hist)
            if np.any(finite):
                sum_surface[k_idx, : hist.size][finite] += hist[finite]
                cnt_surface[k_idx, : hist.size][finite] += 1

    mean_surface = np.full((n_k, max_epoch), np.nan, dtype=float)
    valid = cnt_surface > 0
    mean_surface[valid] = sum_surface[valid] / cnt_surface[valid]
    return mean_surface


def parse_cv_val_surface(cv_running, n_k, path_label=""):
    surface = _mean_surface_from_fold_histories(
        cv_running, history_index=1, n_k=n_k, path_label=path_label
    )
    if surface.shape[1] == 0:
        _warn(f"{path_label}: CV validation histories are empty or missing")
    return surface


def parse_cv_loss_surface(cv_loss_running, n_k, path_label=""):
    return _mean_surface_from_fold_histories(
        cv_loss_running, history_index=1, n_k=n_k, path_label=path_label
    )


def extract_cv_linear_mean_val(cv_acc_linear, n_k):
    vals = np.full((3, n_k), np.nan, dtype=float)
    for fold in range(3):
        for k_idx in range(n_k):
            entry = get_nested_entry(cv_acc_linear, (fold, k_idx))
            vals[fold, k_idx] = extract_scalar_metric(entry, 1)
    with np.errstate(invalid="ignore"):
        return np.nanmean(vals, axis=0)


def select_best_hparams_from_cv(
    model_kind, cv_k_list, cv_val_surface=None, cv_linear_mean_val=None
):
    cv_k_list = np.asarray(cv_k_list, dtype=int)
    if model_kind in {"logistic", "nn"}:
        if cv_val_surface is None or cv_val_surface.size == 0:
            return {
                "selected_K": np.nan,
                "selected_K_idx": None,
                "selected_epoch": np.nan,
                "selected_cv_mean_val_acc": np.nan,
            }
        best = None
        for k_idx, K in enumerate(cv_k_list):
            row = cv_val_surface[k_idx]
            finite_epochs = np.where(np.isfinite(row))[0]
            for epoch in finite_epochs:
                val = row[epoch]
                rank = (float(val), -int(K), -int(epoch))
                if best is None or rank > best["rank"]:
                    best = {
                        "rank": rank,
                        "selected_K": int(K),
                        "selected_K_idx": int(k_idx),
                        "selected_epoch": int(epoch),
                        "selected_cv_mean_val_acc": float(val),
                    }
        if best is None:
            return {
                "selected_K": np.nan,
                "selected_K_idx": None,
                "selected_epoch": np.nan,
                "selected_cv_mean_val_acc": np.nan,
            }
        return best

    if model_kind == "linear":
        if cv_linear_mean_val is None or len(cv_linear_mean_val) == 0:
            return {
                "selected_K": np.nan,
                "selected_K_idx": None,
                "selected_epoch": -1,
                "selected_cv_mean_val_acc": np.nan,
            }
        best = None
        for k_idx, K in enumerate(cv_k_list):
            val = cv_linear_mean_val[k_idx]
            if not np.isfinite(val):
                continue
            rank = (float(val), -int(K))
            if best is None or rank > best["rank"]:
                best = {
                    "rank": rank,
                    "selected_K": int(K),
                    "selected_K_idx": int(k_idx),
                    "selected_epoch": -1,
                    "selected_cv_mean_val_acc": float(val),
                }
        if best is None:
            return {
                "selected_K": np.nan,
                "selected_K_idx": None,
                "selected_epoch": -1,
                "selected_cv_mean_val_acc": np.nan,
            }
        return best

    raise ValueError(model_kind)


def lookup_full_test_at_selected_hparams(model_kind, parsed_run, selected):
    full_k_list = parsed_run["full_K_list"]
    out = {
        "selected_full_train_acc": np.nan,
        "selected_full_test_acc": np.nan,
        "selected_full_test_loss": np.nan,
    }
    K = selected["selected_K"]
    epoch = selected["selected_epoch"]
    if not np.isfinite(K):
        return out

    matches = np.where(full_k_list == int(K))[0]
    if len(matches) == 0:
        _warn(f'{parsed_run["path"]}: selected K={K} not found in full_K_list')
        return out
    full_k_idx = int(matches[0])

    if model_kind in {"logistic", "nn"}:
        full_running = parsed_run["main"]["full_running"]
        full_loss_running = parsed_run["main"]["full_loss"]

        entry = get_nested_entry(full_running, (full_k_idx,))
        train_hist = extract_history(entry, 0)
        test_hist = extract_history(entry, 1)

        if not np.isfinite(epoch):
            return out
        epoch = int(epoch)

        if epoch < len(train_hist):
            out["selected_full_train_acc"] = float(train_hist[epoch])
        else:
            _warn(
                f'{parsed_run["path"]}: selected epoch={epoch} out of range for full train history at K={K}'
            )

        if epoch < len(test_hist):
            out["selected_full_test_acc"] = float(test_hist[epoch])
        else:
            _warn(
                f'{parsed_run["path"]}: selected epoch={epoch} out of range for full test history at K={K}'
            )

        if full_loss_running is not None:
            loss_entry = get_nested_entry(full_loss_running, (full_k_idx,))
            test_loss_hist = extract_history(loss_entry, 1)
            if epoch < len(test_loss_hist):
                out["selected_full_test_loss"] = float(test_loss_hist[epoch])
            elif test_loss_hist.size:
                _warn(
                    f'{parsed_run["path"]}: selected epoch={epoch} out of range for full test loss history at K={K}'
                )

        return out

    if model_kind == "linear":
        entry = get_nested_entry(parsed_run["linear"]["full_acc"], (full_k_idx,))
        out["selected_full_train_acc"] = extract_scalar_metric(entry, 0)
        out["selected_full_test_acc"] = extract_scalar_metric(entry, 1)
        out["selected_full_test_loss"] = np.nan
        return out

    raise ValueError(model_kind)


def parse_run_file(path, expected_kind=None):
    npz_dict = safe_np_load(path)
    keymap = infer_main_keys(npz_dict)
    if keymap is None:
        _warn(f"{path}: cannot infer whether file is logistic or nn; skipping")
        return None
    if expected_kind is not None and keymap["kind"] != expected_kind:
        _warn(
            f"{path}: inferred kind={keymap['kind']} but expected {expected_kind}; skipping"
        )
        return None

    meta = parse_metadata(npz_dict, path)
    cv_k_list = normalize_k_list(npz_dict.get("cv_K_list"))
    full_k_list = normalize_k_list(npz_dict.get("full_K_list"))

    parsed = {
        "path": str(path),
        "model_source": keymap["kind"],
        "seed": meta["seed"],
        "stdz": meta["stdz"],
        "train_mode": meta["train_mode"],
        "N_class": meta["N_class"],
        "num_frames": meta["num_frames"],
        "thresholded": meta["thresholded"],
        "Vmax": meta["Vmax"],
        "cv_K_list": cv_k_list,
        "full_K_list": full_k_list,
        "indices": {
            "cv_train_indices": npz_dict.get("cv_train_indices"),
            "cv_val_indices": npz_dict.get("cv_val_indices"),
            "cv_test_indices": npz_dict.get("cv_test_indices"),
            "full_train_indices": npz_dict.get("full_train_indices"),
            "full_test_indices": npz_dict.get("full_test_indices"),
        },
        "main": {
            "cv_acc": npz_dict.get(keymap["cv_acc_key"]),
            "cv_running": npz_dict.get(keymap["cv_running_key"]),
            "cv_loss": npz_dict.get(keymap["cv_loss_key"]),
            "full_acc": npz_dict.get(keymap["full_acc_key"]),
            "full_running": npz_dict.get(keymap["full_running_key"]),
            "full_loss": npz_dict.get(keymap["full_loss_key"]),
        },
        "linear": {
            "cv_acc": npz_dict.get(keymap["linear_cv_key"]),
            "full_acc": npz_dict.get(keymap["linear_full_key"]),
        },
    }

    n_k_cv = len(cv_k_list)
    parsed["diagnostics"] = {
        "main_cv_val_surface": parse_cv_val_surface(
            parsed["main"]["cv_running"], n_k_cv, path_label=str(path)
        ),
        "main_cv_val_loss_surface": parse_cv_loss_surface(
            parsed["main"]["cv_loss"], n_k_cv, path_label=str(path)
        ),
        "linear_cv_mean_val": extract_cv_linear_mean_val(
            parsed["linear"]["cv_acc"], n_k_cv
        ),
    }
    return parsed


def should_keep_run(parsed):
    if parsed is None:
        return False
    checks = {
        "seed": seed_list,
        "N_class": N_class_list,
        "num_frames": num_frames_list,
        "stdz": stdz_list,
        "train_mode": train_mode_list,
        "thresholded": thresholded_list,
        "Vmax": Vmax_list,
    }
    for field, allowed in checks.items():
        if allowed is None:
            continue
        if parsed[field] not in allowed:
            return False
    return True


def aggregate_curve_items(items, x_name):
    seeds_sorted = sorted({item["seed"] for item in items})
    xs_sorted = np.array(sorted({item[x_name] for item in items}), dtype=int)

    seed_to_idx = {s: i for i, s in enumerate(seeds_sorted)}
    x_to_idx = {x: i for i, x in enumerate(xs_sorted.tolist())}

    selected_K_per_seed = np.full(
        (len(seeds_sorted), len(xs_sorted)), np.nan, dtype=float
    )
    selected_epoch_per_seed = np.full(
        (len(seeds_sorted), len(xs_sorted)), np.nan, dtype=float
    )
    selected_cv_mean_val_acc_per_seed = np.full(
        (len(seeds_sorted), len(xs_sorted)), np.nan, dtype=float
    )
    selected_full_test_acc_per_seed = np.full(
        (len(seeds_sorted), len(xs_sorted)), np.nan, dtype=float
    )
    selected_full_train_acc_per_seed = np.full(
        (len(seeds_sorted), len(xs_sorted)), np.nan, dtype=float
    )
    selected_cv_mean_val_loss_per_seed = np.full(
        (len(seeds_sorted), len(xs_sorted)), np.nan, dtype=float
    )
    selected_full_test_loss_per_seed = np.full(
        (len(seeds_sorted), len(xs_sorted)), np.nan, dtype=float
    )

    for item in items:
        i = seed_to_idx[item["seed"]]
        j = x_to_idx[item[x_name]]
        selected_K_per_seed[i, j] = item["selected_K"]
        selected_epoch_per_seed[i, j] = item["selected_epoch"]
        selected_cv_mean_val_acc_per_seed[i, j] = item["selected_cv_mean_val_acc"]
        selected_full_test_acc_per_seed[i, j] = item["selected_full_test_acc"]
        selected_full_train_acc_per_seed[i, j] = item["selected_full_train_acc"]
        selected_cv_mean_val_loss_per_seed[i, j] = item["selected_cv_mean_val_loss"]
        selected_full_test_loss_per_seed[i, j] = item["selected_full_test_loss"]

    if x_name == "N_class":
        per_seed = {
            seed: {
                "N_class_list": xs_sorted.copy(),
                "selected_K_per_Nclass": selected_K_per_seed[idx].copy(),
                "selected_epoch_per_Nclass": selected_epoch_per_seed[idx].copy(),
                "selected_cv_mean_val_acc_per_Nclass": selected_cv_mean_val_acc_per_seed[
                    idx
                ].copy(),
                "selected_full_test_acc_per_Nclass": selected_full_test_acc_per_seed[
                    idx
                ].copy(),
            }
            for idx, seed in enumerate(seeds_sorted)
        }
    else:
        per_seed = {
            seed: {
                "num_frames_list": xs_sorted.copy(),
                "selected_K_per_shot": selected_K_per_seed[idx].copy(),
                "selected_epoch_per_shot": selected_epoch_per_seed[idx].copy(),
                "selected_cv_mean_val_acc_per_shot": selected_cv_mean_val_acc_per_seed[
                    idx
                ].copy(),
                "selected_full_test_acc_per_shot": selected_full_test_acc_per_seed[
                    idx
                ].copy(),
            }
            for idx, seed in enumerate(seeds_sorted)
        }

    with np.errstate(invalid="ignore"):
        summary = {
            f"{x_name}_list": xs_sorted,
            "seed_list": seeds_sorted,
            "selected_K_per_seed": selected_K_per_seed,
            "selected_epoch_per_seed": selected_epoch_per_seed,
            "selected_cv_mean_val_acc_per_seed": selected_cv_mean_val_acc_per_seed,
            "selected_full_test_acc_per_seed": selected_full_test_acc_per_seed,
            "selected_full_train_acc_per_seed": selected_full_train_acc_per_seed,
            "selected_cv_mean_val_loss_per_seed": selected_cv_mean_val_loss_per_seed,
            "selected_full_test_loss_per_seed": selected_full_test_loss_per_seed,
            "mean_test_acc": np.nanmean(selected_full_test_acc_per_seed, axis=0),
            "std_test_acc": np.nanstd(selected_full_test_acc_per_seed, axis=0),
            "mean_cv_val_acc": np.nanmean(selected_cv_mean_val_acc_per_seed, axis=0),
            "std_cv_val_acc": np.nanstd(selected_cv_mean_val_acc_per_seed, axis=0),
            "per_seed": per_seed,
        }
    return summary


def build_curve_vs_class_count(selection_records):
    grouped = {}
    for rec in selection_records:
        key = (
            rec["stdz"],
            rec["train_mode"],
            rec["num_frames"],
            rec["thresholded"],
            rec["Vmax"],
        )
        grouped.setdefault(key, []).append(rec)
    return {
        key: aggregate_curve_items(items, x_name="N_class")
        for key, items in grouped.items()
    }


def build_curve_vs_shots(selection_records):
    grouped = {}
    for rec in selection_records:
        key = (
            rec["stdz"],
            rec["train_mode"],
            rec["N_class"],
            rec["thresholded"],
            rec["Vmax"],
        )
        grouped.setdefault(key, []).append(rec)
    return {
        key: aggregate_curve_items(items, x_name="num_frames")
        for key, items in grouped.items()
    }


def build_selection_records_from_raw(raw_runs_model, model_kind):
    records = []
    for raw_key, parsed in raw_runs_model.items():
        if model_kind in {"logistic", "nn"}:
            selected = select_best_hparams_from_cv(
                model_kind=model_kind,
                cv_k_list=parsed["cv_K_list"],
                cv_val_surface=parsed["diagnostics"]["main_cv_val_surface"],
            )
            cv_loss_surface = parsed["diagnostics"]["main_cv_val_loss_surface"]
            if (
                selected["selected_K_idx"] is not None
                and np.isfinite(selected["selected_epoch"])
                and cv_loss_surface.ndim == 2
                and selected["selected_K_idx"] < cv_loss_surface.shape[0]
                and int(selected["selected_epoch"]) < cv_loss_surface.shape[1]
            ):
                selected_cv_mean_val_loss = float(
                    cv_loss_surface[
                        selected["selected_K_idx"], int(selected["selected_epoch"])
                    ]
                )
            else:
                selected_cv_mean_val_loss = np.nan
            full_lookup = lookup_full_test_at_selected_hparams(
                model_kind, parsed, selected
            )
        else:
            selected = select_best_hparams_from_cv(
                model_kind="linear",
                cv_k_list=parsed["cv_K_list"],
                cv_linear_mean_val=parsed["diagnostics"]["linear_cv_mean_val"],
            )
            selected_cv_mean_val_loss = np.nan
            full_lookup = lookup_full_test_at_selected_hparams(
                "linear", parsed, selected
            )

        records.append(
            {
                "seed": parsed["seed"],
                "stdz": parsed["stdz"],
                "train_mode": parsed["train_mode"],
                "N_class": parsed["N_class"],
                "num_frames": parsed["num_frames"],
                "thresholded": parsed["thresholded"],
                "Vmax": parsed["Vmax"],
                "selected_K": selected["selected_K"],
                "selected_epoch": selected["selected_epoch"],
                "selected_cv_mean_val_acc": selected["selected_cv_mean_val_acc"],
                "selected_cv_mean_val_loss": selected_cv_mean_val_loss,
                "selected_full_train_acc": full_lookup["selected_full_train_acc"],
                "selected_full_test_acc": full_lookup["selected_full_test_acc"],
                "selected_full_test_loss": full_lookup["selected_full_test_loss"],
                "raw_key": raw_key,
            }
        )
    return records


# ====================
# Scan and parse files
# ====================
raw_runs = {"logistic": {}, "nn": {}, "linear": {}}

for model_kind, result_dir in [
    ("logistic", logistic_result_dir),
    ("nn", nn_result_dir),
]:
    result_dir = Path(result_dir)
    if not result_dir.exists():
        _warn(f"{result_dir} does not exist")
        continue

    for path in sorted(result_dir.glob("*.npz")):
        parsed = parse_run_file(path, expected_kind=model_kind)
        if not should_keep_run(parsed):
            continue

        raw_key = (
            parsed["seed"],
            parsed["stdz"],
            parsed["train_mode"],
            parsed["N_class"],
            parsed["num_frames"],
            parsed["thresholded"],
            parsed["Vmax"],
        )
        if raw_key in raw_runs[model_kind]:
            _warn(f"{path}: duplicate raw_key={raw_key}; overwriting previous entry")
        raw_runs[model_kind][raw_key] = parsed

if preferred_linear_source not in {"logistic", "nn"}:
    raise ValueError("preferred_linear_source must be 'logistic' or 'nn'")

for raw_key, parsed in raw_runs[preferred_linear_source].items():
    raw_runs["linear"][raw_key] = parsed

# ====================
# Build nested-selection summaries
# ====================
results_summary = {
    "logistic": {
        "curve_vs_class_count": {},
        "curve_vs_shots": {},
        "selection_records": [],
    },
    "nn": {"curve_vs_class_count": {}, "curve_vs_shots": {}, "selection_records": []},
    "linear": {
        "curve_vs_class_count": {},
        "curve_vs_shots": {},
        "selection_records": [],
    },
}

for model_kind in ["logistic", "nn", "linear"]:
    selection_records = build_selection_records_from_raw(
        raw_runs[model_kind], model_kind=model_kind
    )
    results_summary[model_kind]["selection_records"] = selection_records
    results_summary[model_kind]["curve_vs_class_count"] = build_curve_vs_class_count(
        selection_records
    )
    results_summary[model_kind]["curve_vs_shots"] = build_curve_vs_shots(
        selection_records
    )

# ====================
# Concise sanity printout
# ====================
print("=" * 80)
print("Raw run counts")
for model_kind in ["logistic", "nn", "linear"]:
    print(f"{model_kind:>8s}: {len(raw_runs[model_kind])}")

print("=" * 80)
print("Summary key counts")
for model_kind in ["logistic", "nn", "linear"]:
    n_class_keys = len(results_summary[model_kind]["curve_vs_class_count"])
    n_shot_keys = len(results_summary[model_kind]["curve_vs_shots"])
    print(
        f"{model_kind:>8s}: curve_vs_class_count={n_class_keys}, curve_vs_shots={n_shot_keys}"
    )

print("=" * 80)
for model_kind in ["logistic", "nn", "linear"]:
    printed = False
    if results_summary[model_kind]["curve_vs_class_count"]:
        key = next(iter(results_summary[model_kind]["curve_vs_class_count"]))
        val = results_summary[model_kind]["curve_vs_class_count"][key]
        print(f"{model_kind} example class-count key: {key}")
        print(
            "  selected_K_per_seed shape =",
            val["selected_K_per_seed"].shape,
            "| mean_test_acc shape =",
            val["mean_test_acc"].shape,
        )
        printed = True
    if (not printed) and results_summary[model_kind]["curve_vs_shots"]:
        key = next(iter(results_summary[model_kind]["curve_vs_shots"]))
        val = results_summary[model_kind]["curve_vs_shots"][key]
        print(f"{model_kind} example shots key: {key}")
        print(
            "  selected_K_per_seed shape =",
            val["selected_K_per_seed"].shape,
            "| mean_test_acc shape =",
            val["mean_test_acc"].shape,
        )
        printed = True
    if not printed:
        print(f"{model_kind}: no summary entries found")

print("=" * 80)
print("Available objects: raw_runs, results_summary")

## Figure 3 (MPEG-7)

In [ ]:
fig_dir = fig_folder_dir / "fig3"
fig_dir.mkdir(parents=True, exist_ok=True)

none_stdz_plot = [
    "eigen"
]  # train_modes in this list use stdz=False; others use stdz=True
fitting_name = {"nn": "NN", "logistic": "Logistic"}

### Image Plot

In [ ]:
scale = 1
tile_gray = 0.88
bg_thresh = 0.92
npz_path = MPEG7_LENS_DIR / "mpeg7_metadata.npz"

class_name_list = [
    "apple",
    "bat",
    "beetle",
    "bell",
    "bird",
    "bone",
    "bottle",
    "brick",
    "butterfly",
]

# layout: 4 tiles + ellipsis + 5 tiles = 10 columns
n_tiles = len(class_name_list)  # 9
ellipsis_col = 4  # 0-based
n_col = n_tiles + 1  # 10

train = np.load(npz_path)["images"]

width = 5.3
fig, ax = plt.subplots(1, n_col, dpi=150, figsize=(width * scale, width / 7 * scale))
fig.patch.set_facecolor("white")


def normalize01(img_raw: np.ndarray) -> np.ndarray:
    img = img_raw.astype(np.float32)
    vmin, vmax = float(img.min()), float(img.max())
    if vmax - vmin < 1e-12:
        return np.zeros_like(img, dtype=np.float32)
    return (img - vmin) / (vmax - vmin)


tile_index = 0
for c in range(n_col):
    a = ax[c]
    a.set_xticks([])
    a.set_yticks([])
    a.set_frame_on(False)

    if c == ellipsis_col:
        # 省略号：无灰底无框
        a.set_facecolor("white")
        a.axis("off")
        a.text(0.5, 0.5, "…", fontsize=20, ha="center", va="center")
        continue

    a.set_title(class_name_list[tile_index], fontsize=6, pad=2)

    img_raw = train[20 * tile_index + 1]
    img01 = normalize01(img_raw)

    if img01.mean() < 0.5:
        img01 = 1.0 - img01

    img_show = img01.copy()
    img_show[img_show > bg_thresh] = tile_gray

    a.imshow(img_show, cmap="gray", vmin=0, vmax=1, interpolation="nearest")

    tile_index += 1

adjust_fixed_margins(fig, left_in=0.0, bottom_in=0, right_in=0, top_in=0)
fig.subplots_adjust(wspace=0.1)
fig.savefig(
    fig_dir / f"mpeg7_features_1row_titles_graytiles.svg",
    format="svg",
    dpi=600,
    bbox_inches=None,
    pad_inches=0.0,
)
plt.show()

### Accuracy vs Class

In [ ]:
scale = 1
thresholded_plot = False
Vmax_plot = True


def get_class_curve_summary(model_kind, train_mode, num_frames, stdz):
    key = (stdz, train_mode, num_frames, thresholded_plot, Vmax_plot)
    curve_dict = results_summary[model_kind]["curve_vs_class_count"]
    if key not in curve_dict:
        available = [
            k for k in curve_dict.keys() if k[1] == train_mode and k[2] == num_frames
        ]
        raise KeyError(
            f"Missing summary for {model_kind}, key={key}. "
            f"Available keys with same train_mode/num_frames: {available[:10]}"
        )
    return curve_dict[key]


for model_kind in ["logistic", "nn"]:
    fig, ax = plt.subplots(figsize=(2.2 * scale, 1.9 * scale))

    num_frames_main = 2
    num_frames_inset = 10

    # -------------------------
    # Main: 2 shots, using nested-selected full-run outer-test accuracy
    # -------------------------
    for train_mode in train_mode_list:
        stdz = (
            False
            if (train_mode in none_stdz_plot and model_kind == "logistic")
            else True
        )
        summary_main = get_class_curve_summary(
            model_kind, train_mode, num_frames_main, stdz
        )

        x_main = np.asarray(summary_main["N_class_list"], dtype=float)
        y_main = np.asarray(summary_main["mean_test_acc"], dtype=float)
        yerr_main = np.asarray(summary_main["std_test_acc"], dtype=float)

        valid = np.isfinite(x_main) & np.isfinite(y_main)
        if np.any(valid):
            ax.errorbar(
                x_main[valid],
                y_main[valid],
                yerr=yerr_main[valid],
                markerfacecolor=color_dic[train_mode] + [face_alpha],
                markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
                c=color_dic[train_mode] + [line_alpha],
                zorder=3,
            )

    ax.set_ylim(55, 105)
    ax.set_yticks(range(50, 101, 10))
    ax.vlines(
        20, *ax.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
    )
    ax.vlines(
        70, *ax.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
    )
    ax.set_xlabel("Number of classes $C$")
    ax.set_ylabel("Test accuracy (%)")
    ax.set_xlim(5, 75)
    ax.grid(alpha=0.8, which="major", linestyle="--", zorder=0)

    title_model = "NN" if model_kind == "nn" else "logistic"
    ax.set_title(f"{fitting_name[model_kind]}, {num_frames_main} shots")
    ax.set_xticks([10, 30, 50, 70])

    # -------------------------
    # Inset: 10 shots, using nested-selected full-run outer-test accuracy
    # -------------------------
    axins = inset_axes(
        ax,
        width="43%",
        height="35%",
        loc="upper right",
        borderpad=0.2,
    )
    axins.set_facecolor("white")
    axins.patch.set_alpha(0.95)

    all_inset_vals = []
    for train_mode in train_mode_list:
        stdz = (
            False
            if (train_mode in none_stdz_plot and model_kind == "logistic")
            else True
        )
        summary_inset = get_class_curve_summary(
            model_kind, train_mode, num_frames_inset, stdz
        )

        x_inset = np.asarray(summary_inset["N_class_list"], dtype=float)
        y_inset = np.asarray(summary_inset["mean_test_acc"], dtype=float)
        yerr_inset = np.asarray(summary_inset["std_test_acc"], dtype=float)

        valid = np.isfinite(x_inset) & np.isfinite(y_inset)
        if np.any(valid):
            all_inset_vals.extend(y_inset[valid].tolist())
            axins.errorbar(
                x_inset[valid],
                y_inset[valid],
                yerr=yerr_inset[valid],
                markersize=1.5 * scale,
                markerfacecolor=color_dic[train_mode] + [face_alpha],
                markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
                c=color_dic[train_mode] + [line_alpha],
                zorder=2,
            )

    if len(all_inset_vals) > 0:
        y0, y1 = float(np.nanmin(all_inset_vals)), float(np.nanmax(all_inset_vals))
        pad = max(1.5, 0.10 * (y1 - y0 + 1e-9))
        axins.set_ylim(y0 - pad, y1 + pad)

        yticks = [round(v / 5) * 5 for v in [y0, (y0 + y1) / 2, y1]]
        yticks = sorted(set(int(t) for t in yticks))
        axins.set_yticks(yticks)

    axins.set_xlim(5, 75)
    axins.vlines(
        20, *axins.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
    )
    axins.vlines(
        70, *axins.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
    )
    axins.set_xticks([10, 30, 50, 70])
    axins.tick_params(axis="both", labelsize=7)
    axins.grid(alpha=0.5, which="major", linestyle="--", zorder=0)

    axins.text(
        0.97,
        0.97,
        f"{num_frames_inset} shots",
        transform=axins.transAxes,
        ha="right",
        va="top",
        fontsize=6,
    )

    adjust_fixed_margins(fig, left_in=0.43, bottom_in=0.35, right_in=0.07, top_in=0.18)
    fig.savefig(
        fig_dir / f"acc_class_{title_model}_inset.svg",
        format="svg",
        dpi=600,
        bbox_inches=None,
        pad_inches=0.0,
    )

    plt.show()

### Accuracy vs Shots

In [ ]:
scale = 1
n_plot_list = [1, 3, 5, 7, 9, 10, 11, 12, 13, 14]
thresholded_plot = False
Vmax_plot = True

requested_num_frames = [num_frames_list[i] for i in n_plot_list]


def get_shot_curve_summary(model_kind, train_mode, n_class, stdz):
    key = (stdz, train_mode, n_class, thresholded_plot, Vmax_plot)
    curve_dict = results_summary[model_kind]["curve_vs_shots"]
    if key not in curve_dict:
        available = [
            k for k in curve_dict.keys() if k[1] == train_mode and k[2] == n_class
        ]
        raise KeyError(
            f"Missing summary for {model_kind}, key={key}. "
            f"Available keys with same train_mode/N_class: {available[:10]}"
        )
    return curve_dict[key]


for model_kind in ["logistic", "nn"]:
    for n_class in [20, 70]:
        fig, ax = plt.subplots(figsize=(2.2 * scale, 1.9 * scale))

        for train_mode in train_mode_list:
            stdz = (
                False
                if (train_mode in none_stdz_plot and model_kind == "logistic")
                else True
            )
            summary = get_shot_curve_summary(model_kind, train_mode, n_class, stdz)

            frames_available = np.asarray(summary["num_frames_list"], dtype=int)
            mean_test_acc = np.asarray(summary["mean_test_acc"], dtype=float)
            std_test_acc = np.asarray(summary["std_test_acc"], dtype=float)

            frame_to_idx = {int(f): i for i, f in enumerate(frames_available.tolist())}

            x_plot, y_plot, yerr_plot = [], [], []
            for nf in requested_num_frames:
                if nf not in frame_to_idx:
                    continue
                idx = frame_to_idx[nf]
                y = mean_test_acc[idx]
                ye = std_test_acc[idx]
                if np.isfinite(y):
                    x_plot.append(nf)
                    y_plot.append(y)
                    yerr_plot.append(ye)

            if len(x_plot) == 0:
                continue

            ax.errorbar(
                np.asarray(x_plot, dtype=float),
                np.asarray(y_plot, dtype=float),
                yerr=np.asarray(yerr_plot, dtype=float),
                markerfacecolor=color_dic[train_mode] + [face_alpha],
                markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
                c=color_dic[train_mode] + [line_alpha],
                zorder=3,
            )

        ax.set_xlabel("Number of shots $S$")
        ax.set_ylabel("Test accuracy (%)")
        ax.set_xlim(0, 30)

        ax.set_ylim(55, 105)
        ax.set_yticks(range(50, 101, 10))

        ax.vlines(
            2, *ax.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
        )
        ax.vlines(
            10, *ax.get_ylim(), colors="0.3", linestyles="--", linewidth=0.5, zorder=0
        )

        ax.grid(alpha=0.8, which="major", linestyle="--", zorder=0)

        title_model = "NN" if model_kind == "nn" else "logistic"
        ax.set_title(f"{fitting_name[model_kind]}, {n_class} classes")

        adjust_fixed_margins(
            fig, left_in=0.43, bottom_in=0.35, right_in=0.07, top_in=0.18
        )
        fig.savefig(
            fig_dir / f"acc_shot_{title_model}_{n_class}_class.svg",
            format="svg",
            dpi=600,
            bbox_inches=None,
            pad_inches=0.0,
        )

        plt.show()

# SPDNN Figures

## Load Data

In [ ]:
# ====================
# User-editable config
# ====================
expts_logistic_dir = RESULT_DIR / "spdnn_expts_logistic/"
expts_nn_dir = RESULT_DIR / "spdnn_expts_nn/"
expts_sim_logistic_dir = RESULT_DIR / "spdnn_expts_sim_logistic/"
expts_sim_nn_dir = RESULT_DIR / "spdnn_expts_sim_nn/"
sim_logistic_dir = RESULT_DIR / "spdnn_sim_logistic/"
sim_nn_dir = RESULT_DIR / "spdnn_sim_nn/"

preferred_linear_source = "logistic"  # "logistic" or "nn"

seed_list = [0, 1, 2, 3, 4]
num_frames_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30]
total_N_list = [50, 100, 200, 300, 400]
stdz_list = [False, True]
train_mode_list = ["lpfft", "eigen", "cg", "pca"]

verbose = True

# ===========
# Directories
# ===========
dir_config = {
    "expts": {
        "logistic": Path(expts_logistic_dir),
        "nn": Path(expts_nn_dir),
    },
    "expts_sim": {
        "logistic": Path(expts_sim_logistic_dir),
        "nn": Path(expts_sim_nn_dir),
    },
    "sim": {
        "logistic": Path(sim_logistic_dir),
        "nn": Path(sim_nn_dir),
    },
}

FILE_RE = re.compile(
    r"^(?P<total_N>\d+)_neurons_(?P<num_frames>\d+)_shots_(?P<train_mode>.+?)_seed_(?P<seed>-?\d+)_stdz_(?P<stdz>True|False)\.npz$"
)

seed_set = None if seed_list is None else set(seed_list)
num_frames_set = None if num_frames_list is None else set(num_frames_list)
total_N_set = None if total_N_list is None else set(total_N_list)
stdz_set = None if stdz_list is None else set(bool(x) for x in stdz_list)
train_mode_set = None if train_mode_list is None else set(train_mode_list)

linear_rank = {"logistic": 1, "nn": 1}
if preferred_linear_source in linear_rank:
    linear_rank[preferred_linear_source] = 0


def safe_np_load(path):
    try:
        with np.load(path, allow_pickle=True) as data:
            return {k: data[k] for k in data.files}
    except Exception as exc:
        warnings.warn(f"Failed to load {path}: {exc}")
        return None


def parse_filename(path):
    match = FILE_RE.match(Path(path).name)
    if match is None:
        return {}
    out = match.groupdict()
    out["total_N"] = int(out["total_N"])
    out["num_frames"] = int(out["num_frames"])
    out["seed"] = int(out["seed"])
    out["stdz"] = out["stdz"] == "True"
    return out


def first_scalar(obj, default=np.nan):
    if obj is None:
        return default
    if isinstance(obj, np.ndarray):
        if obj.ndim == 0:
            return first_scalar(obj.item(), default=default)
        if obj.size == 0:
            return default
        return first_scalar(obj.reshape(-1)[0], default=default)
    if isinstance(obj, (list, tuple)):
        if len(obj) == 0:
            return default
        return first_scalar(obj[0], default=default)
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def coerce_int(obj, default=None):
    try:
        val = first_scalar(obj, default=default)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return default
        return int(val)
    except Exception:
        return default


def coerce_bool(obj, default=None):
    val = first_scalar(obj, default=default)
    if isinstance(val, (bool, np.bool_)):
        return bool(val)
    if isinstance(val, str):
        if val.lower() == "true":
            return True
        if val.lower() == "false":
            return False
    if isinstance(val, (int, np.integer)):
        return bool(val)
    return default


def coerce_str(obj, default=None):
    val = first_scalar(obj, default=default)
    if val is None:
        return default
    return str(val)


def to_1d_float_array(obj, dtype=np.float32):
    if obj is None:
        return np.array([], dtype=dtype)
    if isinstance(obj, np.ndarray):
        if obj.ndim == 0:
            return to_1d_float_array(obj.item(), dtype=dtype)
        if obj.dtype != object:
            try:
                return np.asarray(obj, dtype=dtype).reshape(-1)
            except Exception:
                pass
        vals = []
        for item in obj.reshape(-1):
            vals.extend(to_1d_float_array(item, dtype=np.float64).tolist())
        return np.asarray(vals, dtype=dtype)
    if isinstance(obj, (list, tuple)):
        vals = []
        for item in obj:
            vals.extend(to_1d_float_array(item, dtype=np.float64).tolist())
        return np.asarray(vals, dtype=dtype)
    try:
        return np.asarray([float(obj)], dtype=dtype)
    except Exception:
        return np.array([], dtype=dtype)


def to_1d_int_array(obj, dtype=np.int32):
    arr = to_1d_float_array(obj, dtype=np.float64)
    if arr.size == 0:
        return np.array([], dtype=dtype)
    arr = arr[np.isfinite(arr)]
    return arr.astype(dtype)


def get_entry_by_index(container, idx):
    if container is None:
        return None
    if isinstance(container, np.ndarray):
        if container.ndim == 0:
            return container.item()
        if idx >= len(container):
            return None
        return container[idx]
    if isinstance(container, (list, tuple)):
        if idx >= len(container):
            return None
        return container[idx]
    return None


def parse_final_triplet(obj):
    if obj is None:
        return (np.nan, np.nan, np.nan)
    if isinstance(obj, np.ndarray):
        if obj.ndim == 0:
            return parse_final_triplet(obj.item())
        if obj.dtype != object:
            flat = np.asarray(obj, dtype=np.float64).reshape(-1)
            if flat.size >= 3:
                return (float(flat[0]), float(flat[1]), float(flat[2]))
            if flat.size == 2:
                return (float(flat[0]), np.nan, float(flat[1]))
            if flat.size == 1:
                return (float(flat[0]), np.nan, np.nan)
            return (np.nan, np.nan, np.nan)
        return parse_final_triplet(list(obj))
    if isinstance(obj, (list, tuple)):
        vals = []
        for item in obj[:3]:
            arr = to_1d_float_array(item, dtype=np.float64)
            vals.append(float(arr[0]) if arr.size else np.nan)
        if len(vals) == 2:
            vals = [vals[0], np.nan, vals[1]]
        while len(vals) < 3:
            vals.append(np.nan)
        return tuple(vals[:3])
    try:
        return (float(obj), np.nan, np.nan)
    except Exception:
        return (np.nan, np.nan, np.nan)


def parse_running_triplet(obj):
    empty = (
        np.array([], dtype=np.float32),
        np.array([], dtype=np.float32),
        np.array([], dtype=np.float32),
    )
    if obj is None:
        return empty
    if isinstance(obj, np.ndarray):
        if obj.ndim == 0:
            return parse_running_triplet(obj.item())
        if obj.dtype != object:
            arr = np.asarray(obj, dtype=np.float32)
            if arr.ndim >= 2 and arr.shape[0] == 3:
                return (
                    np.asarray(arr[0], dtype=np.float32).reshape(-1),
                    np.asarray(arr[1], dtype=np.float32).reshape(-1),
                    np.asarray(arr[2], dtype=np.float32).reshape(-1),
                )
            if arr.ndim == 2 and arr.shape[1] == 3:
                return (
                    np.asarray(arr[:, 0], dtype=np.float32).reshape(-1),
                    np.asarray(arr[:, 1], dtype=np.float32).reshape(-1),
                    np.asarray(arr[:, 2], dtype=np.float32).reshape(-1),
                )
            if arr.ndim == 1 and arr.size >= 3:
                return (
                    np.asarray([arr[0]], dtype=np.float32),
                    np.asarray([arr[1]], dtype=np.float32),
                    np.asarray([arr[2]], dtype=np.float32),
                )
            return empty
        return parse_running_triplet(list(obj))
    if isinstance(obj, (list, tuple)):
        if len(obj) >= 3:
            return (
                to_1d_float_array(obj[0], dtype=np.float32),
                to_1d_float_array(obj[1], dtype=np.float32),
                to_1d_float_array(obj[2], dtype=np.float32),
            )
        if len(obj) == 2:
            return (
                to_1d_float_array(obj[0], dtype=np.float32),
                to_1d_float_array(obj[1], dtype=np.float32),
                np.array([], dtype=np.float32),
            )
        if len(obj) == 1:
            return parse_running_triplet(obj[0])
    return empty


def hist_value(hist, epoch_idx):
    if hist is None:
        return np.nan
    arr = to_1d_float_array(hist, dtype=np.float64)
    if epoch_idx is None or epoch_idx < 0 or epoch_idx >= arr.size:
        return np.nan
    return float(arr[epoch_idx])


def select_best_epoch_from_val(val_hist):
    arr = to_1d_float_array(val_hist, dtype=np.float64)
    if arr.size == 0 or np.all(np.isnan(arr)):
        return -1
    best_val = np.nanmax(arr)
    best_idx = np.flatnonzero(arr == best_val)
    return int(best_idx[0]) if best_idx.size else -1


def allowed_setting(run_dict):
    return (
        (seed_set is None or run_dict["seed"] in seed_set)
        and (num_frames_set is None or run_dict["num_frames"] in num_frames_set)
        and (total_N_set is None or run_dict["total_N"] in total_N_set)
        and (stdz_set is None or run_dict["stdz"] in stdz_set)
        and (train_mode_set is None or run_dict["train_mode"] in train_mode_set)
    )


def parse_saved_run(path, dataset_kind, model_kind):
    data = safe_np_load(path)
    if data is None:
        return None, None

    meta_from_name = parse_filename(path)

    total_N = coerce_int(meta_from_name.get("total_N"), default=None)
    num_frames = coerce_int(
        data.get("num_frames"),
        default=coerce_int(meta_from_name.get("num_frames"), default=None),
    )
    train_mode = coerce_str(
        data.get("train_mode"), default=meta_from_name.get("train_mode")
    )
    seed = coerce_int(
        data.get("seed"), default=coerce_int(meta_from_name.get("seed"), default=None)
    )
    stdz = coerce_bool(meta_from_name.get("stdz"), default=None)

    if None in (total_N, num_frames, train_mode, seed) or stdz is None:
        warnings.warn(f"Failed to parse metadata from {path}")
        return None, None

    K_list = to_1d_int_array(data.get("K_list"))
    if K_list.size == 0:
        warnings.warn(f"Empty or missing K_list in {path}")
        return None, None

    acc_key = "acc_logistic" if model_kind == "logistic" else "acc_nn"
    running_key = (
        "acc_logistic_running" if model_kind == "logistic" else "acc_nn_running"
    )
    loss_key = "acc_logistic_loss" if model_kind == "logistic" else "acc_nn_loss"

    final_container = data.get(acc_key)
    running_container = data.get(running_key)
    loss_container = data.get(loss_key)
    linear_container = data.get("acc_linear")

    nK = len(K_list)

    final_train_acc = np.full(nK, np.nan, dtype=np.float32)
    final_val_acc = np.full(nK, np.nan, dtype=np.float32)
    final_test_acc = np.full(nK, np.nan, dtype=np.float32)
    running_acc_triplets = []
    running_loss_triplets = []

    for idx in range(nK):
        f_train, f_val, f_test = parse_final_triplet(
            get_entry_by_index(final_container, idx)
        )
        final_train_acc[idx] = f_train
        final_val_acc[idx] = f_val
        final_test_acc[idx] = f_test
        running_acc_triplets.append(
            parse_running_triplet(get_entry_by_index(running_container, idx))
        )
        running_loss_triplets.append(
            parse_running_triplet(get_entry_by_index(loss_container, idx))
        )

    common_meta = {
        "dataset_kind": dataset_kind,
        "source_model_kind": model_kind,
        "source_path": str(path),
        "source_file": Path(path).name,
        "seed": seed,
        "stdz": stdz,
        "train_mode": train_mode,
        "total_N": total_N,
        "num_frames": num_frames,
        "K_list": K_list.astype(np.int32, copy=False),
        "Vmax": coerce_bool(data.get("Vmax"), default=None),
        "s_train": to_1d_float_array(data.get("s_train"), dtype=np.float32),
        "train_indices": to_1d_int_array(data.get("train_indices")),
        "val_indices": to_1d_int_array(data.get("val_indices")),
        "test_indices": to_1d_int_array(data.get("test_indices")),
    }

    model_run = {
        **common_meta,
        "model_kind": model_kind,
        "final_train_acc_by_K": final_train_acc,
        "final_val_acc_by_K": final_val_acc,
        "final_test_acc_by_K": final_test_acc,
        "running_acc_triplets_by_K": running_acc_triplets,
        "running_loss_triplets_by_K": running_loss_triplets,
    }

    linear_run = None
    if linear_container is not None:
        linear_train_acc = np.full(nK, np.nan, dtype=np.float32)
        linear_val_acc = np.full(nK, np.nan, dtype=np.float32)
        linear_test_acc = np.full(nK, np.nan, dtype=np.float32)
        for idx in range(nK):
            f_train, f_val, f_test = parse_final_triplet(
                get_entry_by_index(linear_container, idx)
            )
            linear_train_acc[idx] = f_train
            linear_val_acc[idx] = f_val
            linear_test_acc[idx] = f_test

        if not np.all(np.isnan(linear_val_acc)):
            linear_run = {
                **common_meta,
                "model_kind": "linear",
                "final_train_acc_by_K": linear_train_acc,
                "final_val_acc_by_K": linear_val_acc,
                "final_test_acc_by_K": linear_test_acc,
                "running_acc_triplets_by_K": None,
                "running_loss_triplets_by_K": None,
            }

    return model_run, linear_run


def select_per_reducedK(run_dict, model_kind):
    K_list = np.asarray(run_dict["K_list"], dtype=np.int32)
    nK = len(K_list)

    epoch_arr = np.full(nK, np.nan, dtype=np.float32)
    val_acc_arr = np.full(nK, np.nan, dtype=np.float32)
    test_acc_arr = np.full(nK, np.nan, dtype=np.float32)
    val_loss_arr = np.full(nK, np.nan, dtype=np.float32)
    test_loss_arr = np.full(nK, np.nan, dtype=np.float32)

    if model_kind == "linear":
        epoch_arr[:] = -1
        val_acc_arr[:] = np.asarray(run_dict["final_val_acc_by_K"], dtype=np.float32)
        test_acc_arr[:] = np.asarray(run_dict["final_test_acc_by_K"], dtype=np.float32)
        return {
            "K_list": K_list,
            "selected_epoch_per_K": epoch_arr,
            "selected_val_acc_per_K": val_acc_arr,
            "selected_test_acc_per_K": test_acc_arr,
            "selected_val_loss_per_K": val_loss_arr,
            "selected_test_loss_per_K": test_loss_arr,
        }

    for idx in range(nK):
        acc_triplet = (
            run_dict["running_acc_triplets_by_K"][idx]
            if idx < len(run_dict["running_acc_triplets_by_K"])
            else None
        )
        loss_triplet = (
            run_dict["running_loss_triplets_by_K"][idx]
            if idx < len(run_dict["running_loss_triplets_by_K"])
            else None
        )

        if acc_triplet is None:
            continue

        _, val_hist, test_hist = acc_triplet
        epoch_idx = select_best_epoch_from_val(val_hist)
        epoch_arr[idx] = epoch_idx
        if epoch_idx >= 0:
            val_acc_arr[idx] = hist_value(val_hist, epoch_idx)
            test_acc_arr[idx] = hist_value(test_hist, epoch_idx)
            if loss_triplet is not None:
                _, val_loss_hist, test_loss_hist = loss_triplet
                val_loss_arr[idx] = hist_value(val_loss_hist, epoch_idx)
                test_loss_arr[idx] = hist_value(test_loss_hist, epoch_idx)

    return {
        "K_list": K_list,
        "selected_epoch_per_K": epoch_arr,
        "selected_val_acc_per_K": val_acc_arr,
        "selected_test_acc_per_K": test_acc_arr,
        "selected_val_loss_per_K": val_loss_arr,
        "selected_test_loss_per_K": test_loss_arr,
    }


def select_best_reducedK_and_epoch(run_dict, model_kind):
    perK = select_per_reducedK(run_dict, model_kind)
    K_list = perK["K_list"]
    val_acc = perK["selected_val_acc_per_K"]

    best_idx = None
    best_val = -np.inf
    best_K = None

    for idx, K in enumerate(K_list):
        v = float(val_acc[idx])
        if not np.isfinite(v):
            continue
        if (
            (best_idx is None)
            or (v > best_val)
            or (v == best_val and int(K) < int(best_K))
        ):
            best_idx = idx
            best_val = v
            best_K = int(K)

    if best_idx is None:
        fallback_idx = int(np.argmin(K_list)) if len(K_list) else -1
        return {
            "selected_reduced_K": (
                float(K_list[fallback_idx]) if fallback_idx >= 0 else np.nan
            ),
            "selected_epoch": -1 if model_kind == "linear" else np.nan,
            "selected_val_acc": np.nan,
            "selected_test_acc": np.nan,
            "selected_val_loss": np.nan,
            "selected_test_loss": np.nan,
        }

    return {
        "selected_reduced_K": float(K_list[best_idx]),
        "selected_epoch": float(perK["selected_epoch_per_K"][best_idx]),
        "selected_val_acc": float(perK["selected_val_acc_per_K"][best_idx]),
        "selected_test_acc": float(perK["selected_test_acc_per_K"][best_idx]),
        "selected_val_loss": float(perK["selected_val_loss_per_K"][best_idx]),
        "selected_test_loss": float(perK["selected_test_loss_per_K"][best_idx]),
    }


def ordered_values(values, preferred=None):
    values = list(values)
    if preferred is None:
        return sorted(values)
    preferred_out = [x for x in preferred if x in set(values)]
    remaining = sorted(set(values) - set(preferred_out))
    return preferred_out + remaining


def nanmean_nanstd(arr2d):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(arr2d, axis=0)
        std = np.nanstd(arr2d, axis=0)
    return np.asarray(mean, dtype=np.float32), np.asarray(std, dtype=np.float32)


def build_curve_vs_shots(model_runs, model_kind):
    grouped = defaultdict(dict)
    for _, run_dict in model_runs.items():
        group_key = (run_dict["stdz"], run_dict["train_mode"], run_dict["total_N"])
        grouped[group_key][(run_dict["seed"], run_dict["num_frames"])] = run_dict

    summary = {}
    for group_key, run_map in grouped.items():
        available_shots = {shot for (_, shot) in run_map.keys()}
        shot_axis = np.asarray(
            ordered_values(available_shots, preferred=num_frames_list), dtype=np.int32
        )
        seed_axis = sorted({seed for (seed, _) in run_map.keys()})

        n_seed = len(seed_axis)
        n_shot = len(shot_axis)

        sel_K = np.full((n_seed, n_shot), np.nan, dtype=np.float32)
        sel_epoch = np.full((n_seed, n_shot), np.nan, dtype=np.float32)
        sel_val_acc = np.full((n_seed, n_shot), np.nan, dtype=np.float32)
        sel_test_acc = np.full((n_seed, n_shot), np.nan, dtype=np.float32)
        sel_val_loss = np.full((n_seed, n_shot), np.nan, dtype=np.float32)
        sel_test_loss = np.full((n_seed, n_shot), np.nan, dtype=np.float32)
        per_seed = {}

        for i_seed, seed in enumerate(seed_axis):
            per_seed_K = np.full(n_shot, np.nan, dtype=np.float32)
            per_seed_epoch = np.full(n_shot, np.nan, dtype=np.float32)
            per_seed_val_acc = np.full(n_shot, np.nan, dtype=np.float32)
            per_seed_test_acc = np.full(n_shot, np.nan, dtype=np.float32)
            per_seed_val_loss = np.full(n_shot, np.nan, dtype=np.float32)
            per_seed_test_loss = np.full(n_shot, np.nan, dtype=np.float32)

            for i_shot, num_frames in enumerate(shot_axis):
                run_dict = run_map.get((seed, int(num_frames)))
                if run_dict is None:
                    continue
                selected = select_best_reducedK_and_epoch(run_dict, model_kind)
                per_seed_K[i_shot] = selected["selected_reduced_K"]
                per_seed_epoch[i_shot] = selected["selected_epoch"]
                per_seed_val_acc[i_shot] = selected["selected_val_acc"]
                per_seed_test_acc[i_shot] = selected["selected_test_acc"]
                per_seed_val_loss[i_shot] = selected["selected_val_loss"]
                per_seed_test_loss[i_shot] = selected["selected_test_loss"]

            sel_K[i_seed] = per_seed_K
            sel_epoch[i_seed] = per_seed_epoch
            sel_val_acc[i_seed] = per_seed_val_acc
            sel_test_acc[i_seed] = per_seed_test_acc
            sel_val_loss[i_seed] = per_seed_val_loss
            sel_test_loss[i_seed] = per_seed_test_loss

            per_seed[seed] = {
                "num_frames_list": shot_axis.copy(),
                "selected_reduced_K_per_shot": per_seed_K,
                "selected_epoch_per_shot": per_seed_epoch,
                "selected_val_acc_per_shot": per_seed_val_acc,
                "selected_test_acc_per_shot": per_seed_test_acc,
                "selected_val_loss_per_shot": per_seed_val_loss,
                "selected_test_loss_per_shot": per_seed_test_loss,
            }

        mean_test_acc, std_test_acc = nanmean_nanstd(sel_test_acc)
        mean_val_acc, std_val_acc = nanmean_nanstd(sel_val_acc)

        summary[group_key] = {
            "epoch_indexing": "0-based",
            "num_frames_list": shot_axis,
            "seed_list": seed_axis,
            "selected_reduced_K_per_seed": sel_K,
            "selected_epoch_per_seed": sel_epoch,
            "selected_val_acc_per_seed": sel_val_acc,
            "selected_test_acc_per_seed": sel_test_acc,
            "selected_val_loss_per_seed": sel_val_loss,
            "selected_test_loss_per_seed": sel_test_loss,
            "mean_test_acc": mean_test_acc,
            "std_test_acc": std_test_acc,
            "mean_val_acc": mean_val_acc,
            "std_val_acc": std_val_acc,
            "per_seed": per_seed,
        }

    return summary


def build_curve_vs_reduced_K(model_runs, model_kind):
    grouped = defaultdict(dict)
    for _, run_dict in model_runs.items():
        group_key = (
            run_dict["stdz"],
            run_dict["train_mode"],
            run_dict["total_N"],
            run_dict["num_frames"],
        )
        grouped[group_key][run_dict["seed"]] = run_dict

    summary = {}
    for group_key, run_map in grouped.items():
        all_K = sorted(
            {int(K) for run_dict in run_map.values() for K in run_dict["K_list"]}
        )
        K_axis = np.asarray(all_K, dtype=np.int32)
        seed_axis = sorted(run_map.keys())

        n_seed = len(seed_axis)
        n_K = len(K_axis)
        K_to_col = {int(K): i for i, K in enumerate(K_axis)}

        sel_epoch = np.full((n_seed, n_K), np.nan, dtype=np.float32)
        sel_val_acc = np.full((n_seed, n_K), np.nan, dtype=np.float32)
        sel_test_acc = np.full((n_seed, n_K), np.nan, dtype=np.float32)
        sel_val_loss = np.full((n_seed, n_K), np.nan, dtype=np.float32)
        sel_test_loss = np.full((n_seed, n_K), np.nan, dtype=np.float32)
        per_seed = {}

        for i_seed, seed in enumerate(seed_axis):
            run_dict = run_map[seed]
            selected = select_per_reducedK(run_dict, model_kind)

            per_seed_epoch = np.full(n_K, np.nan, dtype=np.float32)
            per_seed_val_acc = np.full(n_K, np.nan, dtype=np.float32)
            per_seed_test_acc = np.full(n_K, np.nan, dtype=np.float32)
            per_seed_val_loss = np.full(n_K, np.nan, dtype=np.float32)
            per_seed_test_loss = np.full(n_K, np.nan, dtype=np.float32)

            for local_idx, K in enumerate(selected["K_list"]):
                col = K_to_col[int(K)]
                per_seed_epoch[col] = selected["selected_epoch_per_K"][local_idx]
                per_seed_val_acc[col] = selected["selected_val_acc_per_K"][local_idx]
                per_seed_test_acc[col] = selected["selected_test_acc_per_K"][local_idx]
                per_seed_val_loss[col] = selected["selected_val_loss_per_K"][local_idx]
                per_seed_test_loss[col] = selected["selected_test_loss_per_K"][
                    local_idx
                ]

            sel_epoch[i_seed] = per_seed_epoch
            sel_val_acc[i_seed] = per_seed_val_acc
            sel_test_acc[i_seed] = per_seed_test_acc
            sel_val_loss[i_seed] = per_seed_val_loss
            sel_test_loss[i_seed] = per_seed_test_loss

            per_seed[seed] = {
                "K_list": K_axis.copy(),
                "selected_epoch_per_K": per_seed_epoch,
                "selected_val_acc_per_K": per_seed_val_acc,
                "selected_test_acc_per_K": per_seed_test_acc,
                "selected_val_loss_per_K": per_seed_val_loss,
                "selected_test_loss_per_K": per_seed_test_loss,
            }

        mean_test_acc, std_test_acc = nanmean_nanstd(sel_test_acc)
        mean_val_acc, std_val_acc = nanmean_nanstd(sel_val_acc)

        summary[group_key] = {
            "epoch_indexing": "0-based",
            "K_list": K_axis,
            "seed_list": seed_axis,
            "selected_epoch_per_seed": sel_epoch,
            "selected_val_acc_per_seed": sel_val_acc,
            "selected_test_acc_per_seed": sel_test_acc,
            "selected_val_loss_per_seed": sel_val_loss,
            "selected_test_loss_per_seed": sel_test_loss,
            "mean_test_acc": mean_test_acc,
            "std_test_acc": std_test_acc,
            "mean_val_acc": mean_val_acc,
            "std_val_acc": std_val_acc,
            "per_seed": per_seed,
        }

    return summary


def build_curve_vs_total_K(model_runs, model_kind):
    grouped = defaultdict(dict)
    for _, run_dict in model_runs.items():
        group_key = (run_dict["stdz"], run_dict["train_mode"], run_dict["num_frames"])
        grouped[group_key][(run_dict["seed"], run_dict["total_N"])] = run_dict

    summary = {}
    for group_key, run_map in grouped.items():
        available_total_N = {total_N for (_, total_N) in run_map.keys()}
        total_N_axis = np.asarray(
            ordered_values(available_total_N, preferred=total_N_list), dtype=np.int32
        )
        seed_axis = sorted({seed for (seed, _) in run_map.keys()})

        n_seed = len(seed_axis)
        n_total_N = len(total_N_axis)

        sel_K = np.full((n_seed, n_total_N), np.nan, dtype=np.float32)
        sel_epoch = np.full((n_seed, n_total_N), np.nan, dtype=np.float32)
        sel_val_acc = np.full((n_seed, n_total_N), np.nan, dtype=np.float32)
        sel_test_acc = np.full((n_seed, n_total_N), np.nan, dtype=np.float32)
        sel_val_loss = np.full((n_seed, n_total_N), np.nan, dtype=np.float32)
        sel_test_loss = np.full((n_seed, n_total_N), np.nan, dtype=np.float32)
        per_seed = {}

        for i_seed, seed in enumerate(seed_axis):
            per_seed_K = np.full(n_total_N, np.nan, dtype=np.float32)
            per_seed_epoch = np.full(n_total_N, np.nan, dtype=np.float32)
            per_seed_val_acc = np.full(n_total_N, np.nan, dtype=np.float32)
            per_seed_test_acc = np.full(n_total_N, np.nan, dtype=np.float32)
            per_seed_val_loss = np.full(n_total_N, np.nan, dtype=np.float32)
            per_seed_test_loss = np.full(n_total_N, np.nan, dtype=np.float32)

            for i_total_N, total_N in enumerate(total_N_axis):
                run_dict = run_map.get((seed, int(total_N)))
                if run_dict is None:
                    continue
                selected = select_best_reducedK_and_epoch(run_dict, model_kind)
                per_seed_K[i_total_N] = selected["selected_reduced_K"]
                per_seed_epoch[i_total_N] = selected["selected_epoch"]
                per_seed_val_acc[i_total_N] = selected["selected_val_acc"]
                per_seed_test_acc[i_total_N] = selected["selected_test_acc"]
                per_seed_val_loss[i_total_N] = selected["selected_val_loss"]
                per_seed_test_loss[i_total_N] = selected["selected_test_loss"]

            sel_K[i_seed] = per_seed_K
            sel_epoch[i_seed] = per_seed_epoch
            sel_val_acc[i_seed] = per_seed_val_acc
            sel_test_acc[i_seed] = per_seed_test_acc
            sel_val_loss[i_seed] = per_seed_val_loss
            sel_test_loss[i_seed] = per_seed_test_loss

            per_seed[seed] = {
                "total_N_list": total_N_axis.copy(),
                "selected_reduced_K_per_totalN": per_seed_K,
                "selected_epoch_per_totalN": per_seed_epoch,
                "selected_val_acc_per_totalN": per_seed_val_acc,
                "selected_test_acc_per_totalN": per_seed_test_acc,
                "selected_val_loss_per_totalN": per_seed_val_loss,
                "selected_test_loss_per_totalN": per_seed_test_loss,
            }

        mean_test_acc, std_test_acc = nanmean_nanstd(sel_test_acc)
        mean_val_acc, std_val_acc = nanmean_nanstd(sel_val_acc)

        summary[group_key] = {
            "epoch_indexing": "0-based",
            "total_N_list": total_N_axis,
            "seed_list": seed_axis,
            "selected_reduced_K_per_seed": sel_K,
            "selected_epoch_per_seed": sel_epoch,
            "selected_val_acc_per_seed": sel_val_acc,
            "selected_test_acc_per_seed": sel_test_acc,
            "selected_val_loss_per_seed": sel_val_loss,
            "selected_test_loss_per_seed": sel_test_loss,
            "mean_test_acc": mean_test_acc,
            "std_test_acc": std_test_acc,
            "mean_val_acc": mean_val_acc,
            "std_val_acc": std_val_acc,
            "per_seed": per_seed,
        }

    return summary


# ===========================
# Scan directories and parse
# ===========================
raw_runs = {
    dataset_kind: {"logistic": {}, "nn": {}, "linear": {}}
    for dataset_kind in dir_config
}

for dataset_kind, model_dirs in dir_config.items():
    for model_kind in ("logistic", "nn"):
        run_dir = model_dirs[model_kind]
        if not run_dir.exists():
            warnings.warn(f"Directory does not exist: {run_dir}")
            continue

        file_list = sorted(run_dir.glob("*.npz"))
        if verbose:
            print(
                f"[scan] {dataset_kind}/{model_kind}: {len(file_list)} files in {run_dir}"
            )

        for path in file_list:
            model_run, linear_run = parse_saved_run(
                path, dataset_kind=dataset_kind, model_kind=model_kind
            )

            if model_run is not None and allowed_setting(model_run):
                raw_key = (
                    model_run["seed"],
                    model_run["stdz"],
                    model_run["train_mode"],
                    model_run["total_N"],
                    model_run["num_frames"],
                )
                raw_runs[dataset_kind][model_kind][raw_key] = model_run

            if linear_run is not None and allowed_setting(linear_run):
                raw_key = (
                    linear_run["seed"],
                    linear_run["stdz"],
                    linear_run["train_mode"],
                    linear_run["total_N"],
                    linear_run["num_frames"],
                )
                old_linear = raw_runs[dataset_kind]["linear"].get(raw_key)
                take_new = old_linear is None or linear_rank.get(
                    linear_run["source_model_kind"], 999
                ) < linear_rank.get(old_linear.get("source_model_kind"), 999)
                if take_new:
                    raw_runs[dataset_kind]["linear"][raw_key] = linear_run

# ==================
# Build summary dict
# ==================
results_summary = {
    dataset_kind: {
        "logistic": {},
        "nn": {},
        "linear": {},
    }
    for dataset_kind in dir_config
}

for dataset_kind in raw_runs:
    for model_kind in ("logistic", "nn", "linear"):
        model_runs = raw_runs[dataset_kind][model_kind]
        results_summary[dataset_kind][model_kind] = {
            "curve_vs_shots": build_curve_vs_shots(model_runs, model_kind),
            "curve_vs_reduced_K": build_curve_vs_reduced_K(model_runs, model_kind),
            "curve_vs_total_K": build_curve_vs_total_K(model_runs, model_kind),
        }

# ==============
# Concise report
# ==============
print("\n=== raw run counts ===")
for dataset_kind in ("expts", "expts_sim", "sim"):
    for model_kind in ("logistic", "nn", "linear"):
        print(
            f"{dataset_kind:9s} / {model_kind:8s} : {len(raw_runs[dataset_kind][model_kind])}"
        )

print("\n=== summary key counts ===")
for dataset_kind in ("expts", "expts_sim", "sim"):
    for model_kind in ("logistic", "nn", "linear"):
        block = results_summary[dataset_kind][model_kind]
        print(
            f"{dataset_kind:9s} / {model_kind:8s} : "
            f"vs_shots={len(block['curve_vs_shots'])}, "
            f"vs_reduced_K={len(block['curve_vs_reduced_K'])}, "
            f"vs_total_K={len(block['curve_vs_total_K'])}"
        )

print("\n=== example entries ===")
printed_any = False
for dataset_kind in ("expts", "expts_sim", "sim"):
    for model_kind in ("logistic", "nn", "linear"):
        block = results_summary[dataset_kind][model_kind]
        if block["curve_vs_shots"]:
            example_key = next(iter(block["curve_vs_shots"]))
            example = block["curve_vs_shots"][example_key]
            print(f"{dataset_kind}/{model_kind} curve_vs_shots key = {example_key}")
            print(
                "  selected_test_acc_per_seed shape =",
                example["selected_test_acc_per_seed"].shape,
            )
            print("  mean_test_acc shape =", example["mean_test_acc"].shape)
            printed_any = True
            break
    if printed_any:
        break

printed_any = False
for dataset_kind in ("expts", "expts_sim", "sim"):
    for model_kind in ("logistic", "nn", "linear"):
        block = results_summary[dataset_kind][model_kind]
        if block["curve_vs_reduced_K"]:
            example_key = next(iter(block["curve_vs_reduced_K"]))
            example = block["curve_vs_reduced_K"][example_key]
            print(f"{dataset_kind}/{model_kind} curve_vs_reduced_K key = {example_key}")
            print(
                "  selected_test_acc_per_seed shape =",
                example["selected_test_acc_per_seed"].shape,
            )
            print("  mean_test_acc shape =", example["mean_test_acc"].shape)
            printed_any = True
            break
    if printed_any:
        break

printed_any = False
for dataset_kind in ("expts", "expts_sim", "sim"):
    for model_kind in ("logistic", "nn", "linear"):
        block = results_summary[dataset_kind][model_kind]
        if block["curve_vs_total_K"]:
            example_key = next(iter(block["curve_vs_total_K"]))
            example = block["curve_vs_total_K"][example_key]
            print(f"{dataset_kind}/{model_kind} curve_vs_total_K key = {example_key}")
            print(
                "  selected_test_acc_per_seed shape =",
                example["selected_test_acc_per_seed"].shape,
            )
            print("  mean_test_acc shape =", example["mean_test_acc"].shape)
            printed_any = True
            break
    if printed_any:
        break

## Figure 4 (SPDNN) & NN in Supplementary

In [ ]:
model_kind_list = ["nn", "logistic"]
fig_dir = {
    "logistic": fig_folder_dir / "fig4",
    "nn": fig_folder_dir / "supp_SPDNNwithNN",
}
for model_kind in ["nn", "logistic"]:
    fig_dir[model_kind].mkdir(parents=True, exist_ok=True)

none_stdz_plot = [
    "eigen"
]  # train_modes in this list use stdz=False; others use stdz=True
fitting_name = {"nn": "NN", "logistic": "Logistic"}

In [ ]:
N_list = total_N_list
num_frames = K = 1

for model_kind in model_kind_list:
    fig, ax = plt.subplots(figsize=(3.2 * scale, 2.3 * scale))

    test = "expts"
    for train_mode in train_mode_list:
        stdz = (
            False
            if (train_mode in none_stdz_plot and model_kind == "logistic")
            else True
        )
        key = (stdz, train_mode, num_frames)

        curve_dict = results_summary[test][model_kind]["curve_vs_total_K"]
        if key not in curve_dict:
            warnings.warn(f"Missing key for {test}/{model_kind}: {key}")
            continue

        summary = curve_dict[key]
        x = np.asarray(summary["total_N_list"])
        y = np.asarray(summary["mean_test_acc"])
        yerr = np.asarray(summary["std_test_acc"])

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            markerfacecolor=color_dic[train_mode] + [face_alpha],
            markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
            c=color_dic[train_mode] + [line_alpha],
            label=legend_dic[train_mode],
        )

    ax.hlines(100, xmin=-1, xmax=450, color="gray", ls="--", lw=0.7)
    ax.set_xlabel("Number of neurons $K$")
    ax.set_ylim(78, 98)
    ax.set_yticks([80, 85, 90, 95])
    ax.set_ylabel("Test accuracy (%)")
    ax.set_xlim(0, 450)
    ax.set_xticks([0, 100, 200, 300, 400])
    ax.set_title(
        f"{fitting_name[model_kind]}, {num_frames} shot (experiments, 100 test digits)"
    )
    ax.grid(alpha=0.8, which="major", linestyle="--")

    test = "expts_sim"
    inset1 = fig.add_axes([0.57, 0.25, 0.37, 0.30])

    for train_mode in train_mode_list:
        stdz = False if train_mode in none_stdz_plot else True
        key = (stdz, train_mode, num_frames)

        curve_dict = results_summary[test][model_kind]["curve_vs_total_K"]
        if key not in curve_dict:
            warnings.warn(f"Missing key for {test}/{model_kind}: {key}")
            continue

        summary = curve_dict[key]
        x = np.asarray(summary["total_N_list"])
        y = np.asarray(summary["mean_test_acc"])
        yerr = np.asarray(summary["std_test_acc"])

        inset1.errorbar(
            x,
            y,
            yerr=yerr,
            markerfacecolor=color_dic[train_mode] + [face_alpha],
            markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
            c=color_dic[train_mode] + [line_alpha],
            label=legend_dic[train_mode],
        )

    inset1.grid(alpha=0.8, which="major", linestyle="--")
    inset1.set_xticks(range(0, 500, 100))
    inset1.tick_params(axis="x")
    inset1.tick_params(axis="y")
    inset1.set_ylim(84, 98)
    inset1.set_title(f"simulation, {K} shot")

    adjust_fixed_margins(fig, left_in=0.45, bottom_in=0.35, right_in=0.10, top_in=0.18)
    fig.savefig(
        fig_dir[model_kind] / f"accuracy_N_{num_frames}_shots_{model_kind}.svg",
        format="svg",
        dpi=600,
        bbox_inches=None,
        pad_inches=0.0,
    )
    plt.show()

In [ ]:
scale = 1
n_data = 10
N = 50
num_frames_inset = 1
shots_to_plot = np.array(num_frames_list[:n_data], dtype=int)

for model_kind in model_kind_list:
    fig, ax = plt.subplots(figsize=(3.2 * scale, 2.3 * scale))
    test = "expts"

    # ---------------- main plot: accuracy vs shots ----------------
    for train_mode in train_mode_list:
        stdz = (
            False
            if (train_mode in none_stdz_plot and model_kind == "logistic")
            else True
        )
        key = (stdz, train_mode, N)

        curve_dict = results_summary[test][model_kind]["curve_vs_shots"]
        if key not in curve_dict:
            warnings.warn(f"Missing key for {test}/{model_kind}: {key}")
            continue

        summary = curve_dict[key]
        shot_axis = np.asarray(summary["num_frames_list"], dtype=int)
        y_all = np.asarray(summary["mean_test_acc"], dtype=float)
        yerr_all = np.asarray(summary["std_test_acc"], dtype=float)

        shot_to_idx = {int(s): i for i, s in enumerate(shot_axis)}
        valid_shots = [s for s in shots_to_plot if int(s) in shot_to_idx]

        if len(valid_shots) == 0:
            warnings.warn(f"No requested shots found for {test}/{model_kind}: {key}")
            continue

        idx = [shot_to_idx[int(s)] for s in valid_shots]
        x_main = np.asarray(valid_shots, dtype=int)
        y_main = y_all[idx]
        yerr_main = yerr_all[idx]

        ax.errorbar(
            x_main,
            y_main,
            yerr=yerr_main,
            markerfacecolor=color_dic[train_mode] + [face_alpha],
            markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
            c=color_dic[train_mode] + [line_alpha],
            linestyle="-",
            label=legend_dic[train_mode],
        )

    ax.hlines(100, xmin=-1, xmax=450, color="gray", ls="--", lw=0.7)
    ax.set_xlabel("Number of shots $S$")
    ax.set_ylabel("Test accuracy (%)")
    ax.set_title(
        f"{fitting_name[model_kind]}, {N} neurons (experiments, 100 test digits)"
    )
    ax.set_ylim(78, 98)
    ax.set_yticks([80, 85, 90, 95])
    ax.set_xlim(0, 11)
    ax.grid(alpha=0.8, which="major", linestyle="--")

    # ---------------- inset: accuracy vs reduced K for a fixed shot ----------------
    inset1 = fig.add_axes([0.6, 0.25, 0.34, 0.28])

    for train_mode in train_mode_list:
        stdz = False if train_mode in none_stdz_plot else True
        key = (stdz, train_mode, N, num_frames_inset)

        curve_dict = results_summary[test][model_kind]["curve_vs_reduced_K"]
        if key not in curve_dict:
            warnings.warn(f"Missing key for {test}/{model_kind}: {key}")
            continue

        summary = curve_dict[key]
        x_inset = np.asarray(summary["K_list"], dtype=float)
        y_inset = np.asarray(summary["mean_test_acc"], dtype=float)
        yerr_inset = np.asarray(summary["std_test_acc"], dtype=float)

        inset1.errorbar(
            x_inset,
            y_inset,
            yerr=yerr_inset,
            markerfacecolor=color_dic[train_mode] + [face_alpha],
            markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
            c=color_dic[train_mode] + [line_alpha],
            linestyle="-",
            label=legend_dic[train_mode],
        )

    inset1.grid(alpha=0.8, which="major", linestyle="--")
    inset1.set_xlim(3, 60)
    inset1.set_xscale("log")
    inset1.set_xticks([10, 20, 50, 100])
    inset1.set_xticklabels([r"$10$", r"$20$", r"$50$", r"$K_\mathrm{r}$"])
    inset1.tick_params(axis="x")
    inset1.tick_params(axis="y")
    inset1.set_ylim(80, 90)
    inset1.set_title(f"{num_frames_inset} shot")

    adjust_fixed_margins(fig, left_in=0.44, bottom_in=0.35, right_in=0.10, top_in=0.18)
    fig.savefig(
        fig_dir[model_kind] / f"accuracy_shots_N_{N}_{model_kind}.svg",
        format="svg",
        dpi=600,
        bbox_inches=None,
        pad_inches=0.0,
    )
    plt.show()

## Simulation Results: Logistic and NN

In [ ]:
model_kind_list = ["nn", "logistic"]
fig_dir = {
    "logistic": fig_folder_dir / "supp_SPDNNSim",
    "nn": fig_folder_dir / "supp_SPDNNwithNN",
}
for model_kind in ["nn", "logistic"]:
    fig_dir[model_kind].mkdir(parents=True, exist_ok=True)

none_stdz_plot = [
    "eigen"
]  # train_modes in this list use stdz=False; others use stdz=True
fitting_name = {"nn": "NN", "logistic": "Logistic"}

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

num_frames = K = 1
test = "sim"

for model_kind in ["nn", "logistic"]:
    fig, ax = plt.subplots(figsize=(3.2 * scale, 2.3 * scale))

    for train_mode in train_mode_list:
        stdz = (
            False
            if (train_mode in none_stdz_plot and model_kind == "logistic")
            else True
        )
        key = (stdz, train_mode, num_frames)

        curve_dict = results_summary[test][model_kind]["curve_vs_total_K"]
        if key not in curve_dict:
            warnings.warn(f"Missing key for {test}/{model_kind}: {key}")
            continue

        summary = curve_dict[key]
        x = np.asarray(summary["total_N_list"])
        y = np.asarray(summary["mean_test_acc"])
        yerr = np.asarray(summary["std_test_acc"])

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            markerfacecolor=color_dic[train_mode] + [face_alpha],
            markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
            c=color_dic[train_mode] + [line_alpha],
            label=legend_dic[train_mode],
        )

    ax.hlines(100, xmin=-1, xmax=450, color="gray", ls="--", lw=0.7)
    ax.set_xlabel("Number of neurons $K$")
    ax.set_ylim(83, 96)
    ax.set_ylabel("Test accuracy (%)")
    ax.set_xlim(0, 450)
    ax.set_xticks([0, 100, 200, 300, 400])
    ax.set_title(
        f"{fitting_name[model_kind]}, {num_frames} shot (simulation, 2000 test digits)"
    )
    ax.grid(alpha=0.8, which="major", linestyle="--")

    adjust_fixed_margins(fig, left_in=0.45, bottom_in=0.35, right_in=0.10, top_in=0.18)
    fig.savefig(
        fig_dir[model_kind] / f"accuracy_N_{num_frames}_shots_{model_kind}_sim.svg",
        format="svg",
        dpi=600,
        bbox_inches=None,
        pad_inches=0.0,
    )
    plt.show()

In [ ]:
scale = 1
n_data = 10
N = 50
num_frames_inset = 1
shots_to_plot = np.array(num_frames_list[:n_data], dtype=int)
test = "sim"

for model_kind in ["nn", "logistic"]:
    fig, ax = plt.subplots(figsize=(3.2 * scale, 2.3 * scale))

    # ---------------- main plot: accuracy vs shots ----------------
    for train_mode in train_mode_list:
        stdz = (
            False
            if (train_mode in none_stdz_plot and model_kind == "logistic")
            else True
        )
        key = (stdz, train_mode, N)

        curve_dict = results_summary[test][model_kind]["curve_vs_shots"]
        if key not in curve_dict:
            warnings.warn(f"Missing key for {test}/{model_kind}: {key}")
            continue

        summary = curve_dict[key]
        shot_axis = np.asarray(summary["num_frames_list"], dtype=int)
        y_all = np.asarray(summary["mean_test_acc"], dtype=float)
        yerr_all = np.asarray(summary["std_test_acc"], dtype=float)

        shot_to_idx = {int(s): i for i, s in enumerate(shot_axis)}
        valid_shots = [s for s in shots_to_plot if int(s) in shot_to_idx]

        if len(valid_shots) == 0:
            warnings.warn(f"No requested shots found for {test}/{model_kind}: {key}")
            continue

        idx = [shot_to_idx[int(s)] for s in valid_shots]
        x_main = np.asarray(valid_shots, dtype=int)
        y_main = y_all[idx]
        yerr_main = yerr_all[idx]

        ax.errorbar(
            x_main,
            y_main,
            yerr=yerr_main,
            markerfacecolor=color_dic[train_mode] + [face_alpha],
            markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
            c=color_dic[train_mode] + [line_alpha],
            linestyle="-",
            label=legend_dic[train_mode],
        )

    ax.hlines(100, xmin=-1, xmax=450, color="gray", ls="--", lw=0.7)
    ax.set_xlabel("Number of shots $S$")
    ax.set_ylabel("Test accuracy (%)")
    ax.set_title(
        f"{fitting_name[model_kind]}, {N} neurons (simulation, 2000 test digits)"
    )
    ax.set_ylim(83, 96)
    ax.set_xlim(0, 11)
    ax.grid(alpha=0.8, which="major", linestyle="--")

    # ---------------- inset: accuracy vs reduced K for a fixed shot ----------------
    inset1 = fig.add_axes([0.6, 0.25, 0.34, 0.28])

    for train_mode in train_mode_list:
        stdz = False if train_mode in none_stdz_plot else True
        key = (stdz, train_mode, N, num_frames_inset)

        curve_dict = results_summary[test][model_kind]["curve_vs_reduced_K"]
        if key not in curve_dict:
            warnings.warn(f"Missing key for {test}/{model_kind}: {key}")
            continue

        summary = curve_dict[key]
        x_inset = np.asarray(summary["K_list"], dtype=float)
        y_inset = np.asarray(summary["mean_test_acc"], dtype=float)
        yerr_inset = np.asarray(summary["std_test_acc"], dtype=float)

        inset1.errorbar(
            x_inset,
            y_inset,
            yerr=yerr_inset,
            markerfacecolor=color_dic[train_mode] + [face_alpha],
            markeredgecolor=color_dic[train_mode] + [marker_line_alpha],
            c=color_dic[train_mode] + [line_alpha],
            linestyle="-",
            label=legend_dic[train_mode],
        )

    inset1.grid(alpha=0.8, which="major", linestyle="--")
    inset1.set_xlim(3, 60)
    inset1.set_xscale("log")
    inset1.set_xticks([10, 20, 50, 100])
    inset1.set_xticklabels([r"$10$", r"$20$", r"$50$", r"$K_\mathrm{r}$"])
    inset1.tick_params(axis="x")
    inset1.tick_params(axis="y")
    inset1.set_ylim(70, 90)
    inset1.set_title(f"{num_frames_inset} shot")

    adjust_fixed_margins(fig, left_in=0.44, bottom_in=0.35, right_in=0.10, top_in=0.18)
    fig.savefig(
        fig_dir[model_kind] / f"accuracy_shots_N_{N}_{model_kind}_sim.svg",
        format="svg",
        dpi=600,
        bbox_inches=None,
        pad_inches=0.0,
    )
    plt.show()